# CHB-MIT full-data nested evaluation
Run sections in order on a Colab GPU. This notebook reuses the existing processed data. The default runs one outer fold and one seed; set OUTERS=None for every locked outer fold and SEEDS to all three seeds for the complete study. Each outer fold fits three inner models and one development refit **per variant and seed**.

The main training schedule is now three balanced epochs over the entire eligible pair pool: each class is shuffled, every pair appears at least once per epoch, and minority classes are oversampled to the largest class. This is a new, fixed computational budget, not a demonstrated convergence point. The three variants share initialization and sample order. Training uses current windows and auxiliary next windows; the existing causal history and Top-8 inference rules are retained.

Select the threshold maximizing mean sensitivity across validation subjects with targets, subject to pooled evaluable warning time ≤20%. Ties: lower false alarms/hour, lower warning time, then higher threshold. Zero-target subjects contribute warning burden and false alarms. All locked eligible targets remain in the sensitivity denominator. A threshold above 1 is an explicit no-alert fallback. The 20% cap is a research design choice, not a clinical standard. It is not guaranteed on new patients.

The models are all compared; no model winner is selected. Threshold selection uses inner out-of-fold predictions; each model is then refitted on all development subjects before outer testing. Scores are not calibrated probabilities. Patient uncertainty and a matched chance benchmark are not implemented here; these must accompany scientific claims. Because development data have already informed this protocol, all-subject outer results must be described as exploratory internal validation; chb01 remains the prospectively untouched patient in this workflow.

In [ ]:
%pip -q install numpy pandas h5py matplotlib
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
from google.colab import drive
drive.mount('/content/gdrive')

## 2 · Embedded implementation

In [ ]:
SOURCE_0 = '"""Fold-scoped, causal CHB-MIT loading. No model fitting or alarm scoring."""\nfrom pathlib import Path\nimport json\nimport hashlib\nimport numpy as np\nimport pandas as pd\nimport h5py\n\nclass FoldData:\n    def __init__(self, run, outer=\'outer_chb01\', scope=\'inner_01\'):\n        self.run = Path(run)\n        self.cfg = json.loads((self.run/\'config.json\').read_text())\n        self.inventory = json.loads((self.run/\'locked_inventory.json\').read_text())\n        self.events = json.loads((self.run/\'locked_events.json\').read_text())\n        folds = json.loads((self.run/\'locked_folds.json\').read_text())\n        self.fold = next(f for f in folds if f[\'id\'] == outer)\n        self.outer, self.scope = outer, scope\n        if scope == \'outer_refit\':\n            self.roles = {\'train\': self.fold[\'development_subjects\'], \'test\': self.fold[\'test_subjects\']}\n        else:\n            inner = next(f for f in self.fold[\'inner_folds\'] if f[\'id\'] == scope)\n            self.roles = {\'train\': inner[\'train_subjects\'], \'validation\': inner[\'validation_subjects\']}\n        if set(self.roles[\'train\']) & set(self.fold[\'test_subjects\']):\n            raise ValueError(\'Held-out subject appears in training.\')\n        if len(set(sum(self.roles.values(), []))) != len(sum(self.roles.values(), [])):\n            raise ValueError(\'Roles overlap or contain duplicate subjects.\')\n        status = json.loads((self.run/\'status.json\').read_text())\n        if status.get(\'status\') not in [\'review_required\',\'preprocessing_complete\']:\n            raise ValueError(\'Preprocessing is incomplete.\')\n        blocking = [x for x in status.get(\'cohort_issues\', [])\n                    if x.get(\'reason\') != \'no_valid_preictal_input_after_QC\']\n        if blocking:\n            raise ValueError(f\'Unresolved preprocessing issues: {blocking}\')\n        if status.get(\'cohort_issues\'):\n            audit = json.loads((self.run/\'training_label_audit_status_v2_2.json\').read_text())\n            if audit.get(\'revision\') != \'2.2\':\n                raise ValueError(\'Run preprocessing Section 7B first.\')\n        self.scaler_path = self.run/\'folds\'/outer/scope/\'train_scaler.npz\'\n        with np.load(self.scaler_path, allow_pickle=False) as z:\n            if set(z[\'training_subjects\'].tolist()) != set(self.roles[\'train\']):\n                raise ValueError(\'Scaler training subjects do not match the fold.\')\n            if set(z[\'test_subjects\'].tolist()) != set(self.fold[\'test_subjects\']):\n                raise ValueError(\'Scaler outer test subjects do not match.\')\n            self.scaler = {k:z[k].copy() for k in [\'raw_mean\',\'raw_std\',\'feature_mean\',\'feature_std\']}\n        C = len(self.cfg[\'channels\'])\n        for k,shape in [(\'raw_mean\',(C,)),(\'raw_std\',(C,)),(\'feature_mean\',(C,5)),(\'feature_std\',(C,5))]:\n            v = self.scaler[k]\n            if v.shape != shape or not np.isfinite(v).all() or (\'std\' in k and np.any(v<=0)):\n                raise ValueError(f\'Invalid scaler: {k}\')\n        ratio = self.cfg[\'fusion_history_s\']/self.cfg[\'window_s\']\n        if ratio != int(ratio):\n            raise ValueError(\'Fusion history must contain whole windows.\')\n        self.history = int(ratio)\n        self.subject_index = {s:i for i,s in enumerate(sorted({r[\'subject\'] for r in self.inventory}))}\n        self._metadata = {}\n\n    def records(self, role):\n        if role not in self.roles:\n            raise ValueError(f\'{role} is unavailable in {self.scope}; allowed: {list(self.roles)}\')\n        rows = [r for r in self.inventory if r[\'subject\'] in self.roles[role]\n                and r[\'signal_ok\'] and not r[\'excluded_case\'] and r[\'split\']==\'pool\']\n        return sorted(rows, key=lambda r:(r[\'subject\'],r[\'case\'],r[\'start_s\'],r[\'relative_path\']))\n\n    def path(self, row):\n        return self.run/\'processed\'/row[\'case\']/\'pool\'/(Path(row[\'relative_path\']).stem+\'.h5\')\n\n    def metadata(self, row):\n        p = self.path(row)\n        if p not in self._metadata:\n            with h5py.File(p,\'r\') as h:\n                if str(h.attrs[\'case\']) != row[\'case\'] or json.loads(h.attrs[\'channels_json\']) != self.cfg[\'channels\']:\n                    raise ValueError(f\'Unexpected case or channels: {p}\')\n                if json.loads(h.attrs[\'config_json\']) != self.cfg:\n                    raise ValueError(f\'Unexpected processing configuration: {p}\')\n                keys = [\'label\',\'valid_signal\',\'window_start_s\',\'decision_s\',\'pair_with_previous\']\n                m = {k:h[k][:] for k in keys}\n            n = len(m[\'label\']); w = self.cfg[\'window_s\']\n            if any(len(v)!=n for v in m.values()) or not np.isin(m[\'label\'],[-1,0,1,2]).all():\n                raise ValueError(f\'Invalid metadata: {p}\')\n            if n and (not np.allclose(m[\'decision_s\']-m[\'window_start_s\'],w) or\n                      np.any(np.diff(m[\'decision_s\'])<=0)):\n                raise ValueError(f\'Invalid timeline: {p}\')\n            # History ends before current window starts. Reset on every EDF boundary,\n            # missing interval, or invalid signal; never concatenate across files.\n            ready = np.zeros(n,dtype=bool)\n            streak = 0\n            for i in range(n):\n                if i and not np.isclose(m[\'window_start_s\'][i],m[\'decision_s\'][i-1],rtol=0,atol=1e-6):\n                    streak = 0\n                if m[\'valid_signal\'][i]:\n                    ready[i] = streak >= self.history\n                    streak += 1\n                else:\n                    streak = 0\n            m[\'prediction_ready\'] = ready\n            self._metadata[p] = m\n        return self._metadata[p]\n\n    def index(self, role, mode=\'supervised\'):\n        if mode not in [\'supervised\',\'pairs\',\'replay\']:\n            raise ValueError(mode)\n        if mode==\'pairs\' and role!=\'train\':\n            raise ValueError(\'Future-window pairs are restricted to training.\')\n        items=[]\n        for row in self.records(role):\n            m=self.metadata(row); n=len(m[\'label\'])\n            if mode==\'replay\': keep=np.ones(n,dtype=bool)\n            else:\n                keep=m[\'prediction_ready\'] & (m[\'label\']>=0)\n                if mode==\'pairs\':\n                    next_ok=np.zeros(n,dtype=bool)\n                    if n>1:\n                        next_ok[:-1]=m[\'valid_signal\'][1:] & m[\'pair_with_previous\'][1:]\n                    keep &= next_ok\n            items.extend((row,int(i)) for i in np.flatnonzero(keep))\n        return items\n\n    def read(self, row, i, mode=\'supervised\'):\n        if mode not in [\'supervised\',\'pairs\',\'replay\']: raise ValueError(mode)\n        m=self.metadata(row); ready=bool(m[\'prediction_ready\'][i])\n        if mode!=\'replay\' and (not ready or m[\'label\'][i]<0):\n            raise ValueError(\'Supervised sample lacks labels or causal history.\')\n        C=len(self.cfg[\'channels\']); L=int(self.cfg[\'fs\']*self.cfg[\'window_s\']); H=self.history\n        inputs={\'raw_t\':np.zeros((C,L),np.float32), \'spec_t\':np.zeros((C,5),np.float32),\n                \'raw_past\':np.zeros((H,C,L),np.float32), \'spec_past\':np.zeros((H,C,5),np.float32)}\n        def normal(raw,spec):\n            s=self.scaler\n            x=((raw-s[\'raw_mean\'][:,None])/s[\'raw_std\'][:,None]).astype(np.float32)\n            f=((spec-s[\'feature_mean\'])/s[\'feature_std\']).astype(np.float32)\n            if not np.isfinite(x).all() or not np.isfinite(f).all():\n                raise ValueError(\'Nonfinite normalized input\')\n            return x,f\n        auxiliary={}\n        with h5py.File(self.path(row),\'r\') as h:\n            # Unavailable replay rows retain timestamps but carry zero placeholders.\n            # The model must run ONLY when prediction_ready is true.\n            if ready:\n                inputs[\'raw_t\'],inputs[\'spec_t\']=normal(h[\'raw_uv\'][i],h[\'features\'][i])\n                inputs[\'raw_past\'],inputs[\'spec_past\']=normal(h[\'raw_uv\'][i-H:i],h[\'features\'][i-H:i])\n            if mode==\'pairs\':\n                if row[\'subject\'] not in self.roles[\'train\']:\n                    raise ValueError(\'Future-window pairs are restricted to training subjects.\')\n                if i+1>=len(m[\'label\']) or not m[\'valid_signal\'][i+1] or not m[\'pair_with_previous\'][i+1]:\n                    raise ValueError(\'Invalid adjacent pair.\')\n                auxiliary[\'raw_tdt\'],auxiliary[\'spec_tdt\']=normal(h[\'raw_uv\'][i+1],h[\'features\'][i+1])\n        return dict(inputs=inputs, auxiliary=auxiliary,\n                    target=dict(label=int(m[\'label\'][i])),\n                    metadata=dict(subject=row[\'subject\'],case=row[\'case\'],patient_idx=self.subject_index[row[\'subject\']],\n                                  file=str(self.path(row)),index=i,start_s=float(m[\'window_start_s\'][i]),\n                                  decision_s=float(m[\'decision_s\'][i]),valid_signal=bool(m[\'valid_signal\'][i]),\n                                  prediction_ready=ready, reset_at_file_start=i==0))\n\n    def iterate(self, role, mode=\'replay\', shuffle=False, seed=20261028):\n        if shuffle and (role!=\'train\' or mode==\'replay\'):\n            raise ValueError(\'Only supervised training may be shuffled.\')\n        items=self.index(role,mode)\n        if shuffle:\n            rng=np.random.default_rng(seed); rng.shuffle(items)\n        for row,i in items: yield self.read(row,i,mode)\n\n    def audit(self, role):\n        rows=[];targets=[]\n        for row in self.records(role):\n            m=self.metadata(row)\n            rows.append(dict(subject=row[\'subject\'],file=row[\'relative_path\'],windows=len(m[\'label\']),\n                             valid_signal=int(m[\'valid_signal\'].sum()),prediction_ready=int(m[\'prediction_ready\'].sum()),\n                             **{f\'ready_label_{k}\':int((m[\'prediction_ready\']&(m[\'label\']==k)).sum()) for k in [-1,0,1,2]}))\n        for e in self.events:\n            if not e[\'eligible_target\'] or e[\'split\']!=\'pool\' or e[\'subject\'] not in self.roles[role]:continue\n            a=e[\'onset_s\']-self.cfg[\'sph_s\']-self.cfg[\'sop_s\'];b=e[\'onset_s\']-self.cfg[\'sph_s\']\n            n=0\n            for row in self.records(role):\n                if row[\'case\']!=e[\'case\']:continue\n                m=self.metadata(row)\n                # Decision times, independent of supervised labels.\n                n+=int((m[\'prediction_ready\']&(m[\'decision_s\']>=a)&(m[\'decision_s\']<=b)).sum())\n            targets.append(dict(event_id=e[\'event_id\'],subject=e[\'subject\'],retain_in_evaluation=True,\n                                available_forecast_decisions=n,has_forecast_opportunity=n>0))\n        return pd.DataFrame(rows),pd.DataFrame(targets)\n'
exec(compile(SOURCE_0, "embedded_0.py", "exec"))

In [ ]:
SOURCE_1 = 'import numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass CoreMVModel(nn.Module):\n    """Explicit smoke-test foundation, not the complete physics-informed model."""\n    def __init__(self, channels=18, latent=32, rank=8):\n        super().__init__()\n        self.raw_conv=nn.Sequential(nn.Conv1d(channels,channels,7,padding=3,groups=channels),nn.GELU(),\n                                    nn.Conv1d(channels,64,1),nn.GELU())\n        self.raw_project=nn.Linear(128,latent)\n        self.spectral=nn.Sequential(nn.Flatten(1),nn.Linear(channels*5,64),nn.LayerNorm(64),nn.GELU(),nn.Linear(64,latent))\n        self.fusion=nn.Sequential(nn.Linear(latent*2,latent),nn.LayerNorm(latent),nn.GELU())\n        self.classifier=nn.Linear(latent,3)\n        self.U=nn.Parameter(torch.randn(latent,rank)*0.01)\n        self.V=nn.Parameter(torch.randn(latent,rank)*0.01)\n\n    def encode(self,raw,spec):\n        h=self.raw_conv(raw)\n        zr=self.raw_project(torch.cat([h.mean(-1),h.amax(-1)],dim=-1))\n        zs=self.spectral(spec)\n        return self.fusion(torch.cat([zr,zs],dim=-1))\n\n    def propagate(self,z):\n        # Row-vector convention for K=I+UV^T: z_next = z K^T.\n        return z+(z@self.V)@self.U.T\n\n    def forward(self,raw,spec):\n        z=self.encode(raw,spec)\n        return self.classifier(z),z\n\n    def operator_matrix(self):\n        return torch.eye(self.U.shape[0],device=self.U.device)+self.U@self.V.T\n\n\ndef tensor_batch(samples,device,paired=False):\n    def stack(group,key):\n        return torch.as_tensor(np.stack([s[group][key] for s in samples]),dtype=torch.float32,device=device)\n    out={\'raw\':stack(\'inputs\',\'raw_t\'),\'spec\':stack(\'inputs\',\'spec_t\'),\n         \'labels\':torch.tensor([s[\'target\'][\'label\'] for s in samples],dtype=torch.long,device=device)}\n    if paired:\n        out.update(next_raw=stack(\'auxiliary\',\'raw_tdt\'),next_spec=stack(\'auxiliary\',\'spec_tdt\'))\n    return out\n\n\ndef core_loss(model,batch,dynamics_weight=0.5):\n    logits,z=model(batch[\'raw\'],batch[\'spec\'])\n    ce=F.cross_entropy(logits,batch[\'labels\'])\n    with torch.no_grad():\n        target=model.encode(batch[\'next_raw\'],batch[\'next_spec\'])\n    dynamics=F.mse_loss(model.propagate(z),target)\n    return ce+dynamics_weight*dynamics,{\'classification\':ce,\'dynamics\':dynamics},z\n\n\ndef check_gradients(model):\n    report={}\n    for name,p in model.named_parameters():\n        if p.grad is None or not torch.isfinite(p.grad).all():\n            raise RuntimeError(f\'Missing/nonfinite gradient: {name}\')\n        report[name]=float(p.grad.norm().detach().cpu())\n    if not any(x>0 for x in report.values()): raise RuntimeError(\'All gradients are zero\')\n    return report\n\n\n@torch.no_grad()\ndef causal_probabilities(model, sample, device, top_k=8, encode_chunk=16):\n    if not sample[\'metadata\'][\'prediction_ready\']:\n        return None\n    inputs=sample[\'inputs\']\n    # Manuscript convention: last 74 preceding windows + current = 75.\n    # Loader requires 75 preceding valid windows, so its availability rule is\n    # conservatively one window stricter. Preserve it; never silently relax it.\n    H=len(inputs[\'raw_past\'])\n    if H<top_k or H<1: raise ValueError(\'Insufficient history for Top-K score\')\n    raw=np.concatenate([inputs[\'raw_past\'][1:],inputs[\'raw_t\'][None]],axis=0)\n    spec=np.concatenate([inputs[\'spec_past\'][1:],inputs[\'spec_t\'][None]],axis=0)\n    probabilities=[]\n    was_training=model.training;model.eval()\n    try:\n        for start in range(0,len(raw),encode_chunk):\n            logits,_=model(torch.as_tensor(raw[start:start+encode_chunk],device=device),\n                           torch.as_tensor(spec[start:start+encode_chunk],device=device))\n            probabilities.append(logits.softmax(-1).cpu())\n    finally:\n        model.train(was_training)\n    p=torch.cat(probabilities).numpy()\n    return dict(p_interictal=float(p[-1,0]),p_preictal=float(p[-1,1]),p_ictal=float(p[-1,2]),\n                top8_preictal=float(np.sort(p[:,1])[-top_k:].mean()))\n\n\ndef smoke_sample_indices(data,per_class=2,seed=20261028):\n    # Deliberately tiny class-stratified engineering test, not a training sampler.\n    candidates=data.index(\'train\',\'pairs\');rng=np.random.default_rng(seed)\n    picked=[]\n    for label in [0,1,2]:\n        eligible=[x for x in candidates if data.metadata(x[0])[\'label\'][x[1]]==label]\n        if not eligible: raise RuntimeError(f\'No class {label} dynamics pairs in this training split\')\n        chosen=rng.choice(len(eligible),size=min(per_class,len(eligible)),replace=False)\n        picked.extend(eligible[int(i)] for i in chosen)\n    return picked\n'
exec(compile(SOURCE_1, "embedded_1.py", "exec"))

In [ ]:
SOURCE_2 = 'import math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nclass WCInspired(nn.Module):\n    """16 uncoupled formal E/I pairs sharing 8 parameters; effective seconds.\n    Reduced sigmoid rate model, not the full original Wilson-Cowan equations.\n    """\n    def __init__(self):\n        super().__init__()\n        self.weight_logits=nn.Parameter(torch.logit(torch.tensor([1.5,1.,1.,0.5])/4))\n        self.tau_logits=nn.Parameter(torch.logit((torch.tensor([10.,16.])-4)/36))\n        self.drive_logits=nn.Parameter(torch.atanh(torch.tensor([-.5,-.5])/2))\n\n    def physical_parameters(self):\n        return 4*torch.sigmoid(self.weight_logits),4+36*torch.sigmoid(self.tau_logits),2*torch.tanh(self.drive_logits)\n\n    def activation(self,state):\n        E,I=state.chunk(2,dim=-1)\n        w,tau,b=self.physical_parameters()\n        return torch.cat([torch.sigmoid(w[0]*E-w[1]*I+b[0]),\n                          torch.sigmoid(w[2]*E-w[3]*I+b[1])],dim=-1)\n\n    def rhs(self,state):\n        _,tau,_=self.physical_parameters()\n        rate=torch.cat([tau[0].expand(state.shape[-1]//2),tau[1].expand(state.shape[-1]//2)])\n        return (self.activation(state)-state)/rate\n\n    def forward(self,state,dt_s=4.,substeps=16):\n        if state.shape[-1]!=32:raise ValueError(\'Expected 16 E + 16 I coordinates\')\n        if substeps<1 or int(substeps)!=substeps or not math.isfinite(dt_s) or dt_s<=0:\n            raise ValueError(\'Invalid integration settings\')\n        if not torch.isfinite(state).all() or torch.any(state<0) or torch.any(state>1):\n            raise ValueError(\'Initial state must be finite and in [0,1]\')\n        _,tau,_=self.physical_parameters()\n        rate=torch.cat([tau[0].expand(16),tau[1].expand(16)])\n        # Exponential Euler: exact leak, activation frozen per substep.\n        # Convex combination preserves [0,1] without clipping. This is NOT\n        # an assertion of global dynamical stability or numerical convergence.\n        decay=torch.exp(-(dt_s/int(substeps))/rate)\n        y=state\n        for _ in range(int(substeps)):\n            y=decay*y+(1-decay)*self.activation(y)\n        return y\n\nclass PhysicsMVModel(CoreMVModel):\n    def __init__(self,channels=18,latent=32,rank=8):\n        if latent!=32:raise ValueError(\'Physics v1 requires latent=32\')\n        super().__init__(channels,latent,rank)\n        self.wc=WCInspired()\n\n\ndef physics_loss(model,batch,dt_s=4.,substeps=16,dynamics_weight=.5,physics_weight=.2):\n    logits,z=model(batch[\'raw\'],batch[\'spec\'])\n    with torch.no_grad():\n        next_z=model.encode(batch[\'next_raw\'],batch[\'next_spec\'])\n        next_state=torch.sigmoid(next_z)\n    kz=model.propagate(z)\n    wc_state=model.wc(torch.sigmoid(z),dt_s=dt_s,substeps=substeps)\n    classification=F.cross_entropy(logits,batch[\'labels\'])\n    dynamics=F.mse_loss(kz,next_z)\n    coupling=F.mse_loss(torch.sigmoid(kz),wc_state)\n    empirical=F.mse_loss(wc_state,next_state)\n    physics=(coupling+empirical)/2\n    total=classification+dynamics_weight*dynamics+physics_weight*physics\n    return total,dict(classification=classification,dynamics=dynamics,\n                      physics_coupling=coupling,physics_empirical=empirical,physics=physics),z\n\n@torch.no_grad()\ndef physics_diagnostics(model,batch,dt_s=4.):\n    z=model.encode(batch[\'raw\'],batch[\'spec\'])\n    state=torch.sigmoid(z)\n    y16=model.wc(state,dt_s,16);y32=model.wc(state,dt_s,32);y256=model.wc(state,dt_s,256)\n    centered=z.double()-z.double().mean(0)\n    eigen=torch.linalg.svdvals(centered).square()\n    if eigen.sum()>1e-20:\n        p=eigen/eigen.sum();p=p[p>0];rank=float(torch.exp(-(p*p.log()).sum()).cpu())\n    else:rank=0.\n    K=model.operator_matrix().detach().double().cpu()\n    w,tau,b=model.wc.physical_parameters()\n    return dict(state_min=float(y16.min().cpu()),state_max=float(y16.max().cpu()),\n                max_difference_16_vs_32=float((y16-y32).abs().max().cpu()),\n                max_difference_16_vs_256=float((y16-y256).abs().max().cpu()),\n                max_difference_32_vs_256=float((y32-y256).abs().max().cpu()),\n                latent_mean_std=float(z.std(0,unbiased=False).mean().cpu()),\n                latent_effective_rank=rank,effective_rank_upper_bound=min(len(z)-1,z.shape[-1]),\n                state_saturated_fraction=float(((state<.01)|(state>.99)).float().mean().cpu()),\n                koopman_spectral_radius=float(torch.linalg.eigvals(K).abs().max()),\n                koopman_largest_singular_value=float(torch.linalg.svdvals(K).max()),\n                wc_weights=w.cpu().tolist(),wc_tau_effective_seconds=tau.cpu().tolist(),wc_drives=b.cpu().tolist())\n'
exec(compile(SOURCE_2, "embedded_2.py", "exec"))

In [ ]:
SOURCE_3 = '"""Fixed research scoring protocol v1. No training or threshold optimization."""\nfrom dataclasses import dataclass, asdict\nimport numpy as np\nimport pandas as pd\n\n@dataclass(frozen=True)\nclass ScoringPolicy:\n    sph_s: float = 300.0\n    sop_s: float = 1800.0\n    decision_bin_s: float = 4.0\n    threshold: float = 0.5  # Synthetic demonstration only; not a selected operating point.\n\n    def __post_init__(self):\n        if self.sph_s<0 or self.sop_s<=0 or self.decision_bin_s<=0 or not 0<=self.threshold<=1.01:\n            raise ValueError(\'Invalid scoring policy\')\n\n\ndef merge_intervals(intervals):\n    out=[]\n    for a,b in sorted((float(a),float(b)) for a,b in intervals):\n        if not np.isfinite([a,b]).all() or b<=a: raise ValueError(\'Invalid coverage interval\')\n        if out and a<=out[-1][1]:out[-1]=(out[-1][0],max(out[-1][1],b))\n        else:out.append((a,b))\n    return out\n\n\ndef fully_covered(a,b,intervals):\n    return any(x<=a and y>=b for x,y in intervals)\n\n\ndef covered_many(starts, ends, intervals):\n    starts=np.asarray(starts,dtype=float);ends=np.asarray(ends,dtype=float)\n    if not intervals:return np.zeros(starts.shape,dtype=bool)\n    a=np.array([x[0] for x in intervals]);b=np.array([x[1] for x in intervals])\n    idx=np.searchsorted(a,starts,side=\'right\')-1\n    safe=np.maximum(idx,0)\n    return (idx>=0)&(b[safe]>=ends)\n\n\ndef score_case(predictions,events,coverage,policy=ScoringPolicy()):\n    """One case, one relative-time origin. Future annotations label outcomes only.\n\n    predictions: ALL stored decision rows, including unavailable rows. Required:\n    decision_s, prediction_ready(bool), score(float, NaN when unavailable).\n    events: event_id, onset_s, offset_s, eligible_target(bool).\n    coverage: annotation-known intervals, including montage-unusable recordings.\n    Exposure uses discrete decision bins, not interpolation over signal gaps.\n    """\n    p=predictions.copy();e=events.copy();obs=merge_intervals(coverage)\n    required={\'decision_s\',\'prediction_ready\',\'score\'}\n    if not required.issubset(p):raise ValueError(f\'Missing prediction fields: {required-set(p.columns)}\')\n    if not {\'event_id\',\'onset_s\',\'offset_s\',\'eligible_target\'}.issubset(e):raise ValueError(\'Missing event fields\')\n    if not p.prediction_ready.isin([True,False]).all() or not e.eligible_target.isin([True,False]).all():\n        raise ValueError(\'Availability and eligibility must be boolean\')\n    p=p.sort_values(\'decision_s\').reset_index(drop=True)\n    t=p.decision_s.to_numpy(dtype=float);ready=p.prediction_ready.to_numpy(dtype=bool)\n    scores=p.score.to_numpy(dtype=float)\n    if not np.isfinite(t).all() or (len(t)>1 and np.any(np.diff(t)<policy.decision_bin_s-1e-6)):\n        raise ValueError(\'Duplicate, overlapping, or invalid decision times\')\n    if not np.isfinite(scores[ready]).all() or np.any((scores[ready]<0)|(scores[ready]>1)):\n        raise ValueError(\'Ready scores must be finite probabilities in [0,1]\')\n    if np.any(np.isfinite(scores[~ready])):raise ValueError(\'Unavailable scores must be NaN\')\n    if e.event_id.duplicated().any() or not np.isfinite(e[[\'onset_s\',\'offset_s\']].to_numpy(dtype=float)).all() or np.any(e.offset_s<=e.onset_s):\n        raise ValueError(\'Invalid event table\')\n    if not covered_many(t-policy.decision_bin_s,t,obs).all():\n        raise ValueError(\'Decision bin is outside annotated recording coverage\')\n    for row in e.itertuples():\n        if not fully_covered(row.onset_s,row.offset_s,obs):\n            raise ValueError(\'Event lies outside known annotations\')\n    H=policy.sph_s+policy.sop_s\n    evaluable=covered_many(t,t+H,obs)&ready\n    alarms=[];next_allowed=-np.inf\n    # Alarm generation uses scores, readiness, time and fixed cooldown only.\n    # It never reads eligibility, seizure labels or future coverage.\n    for i,x in enumerate(t):\n        if not ready[i] or scores[i]<policy.threshold or x<next_allowed:continue\n        alarm_id=len(alarms);a=x+policy.sph_s;b=x+H\n        within=e[(e.onset_s>a)&(e.onset_s<=b)]\n        target_ids=within.loc[within.eligible_target.astype(bool),\'event_id\'].tolist()\n        non_target_ids=within.loc[~within.eligible_target.astype(bool),\'event_id\'].tolist()\n        known=fully_covered(x,b,obs)\n        outcome=(\'target_hit\' if target_ids else \'non_target_seizure\' if non_target_ids else\n                 \'false_alarm\' if known else \'unscorable\')\n        alarms.append(dict(alarm_id=alarm_id,alarm_s=x,sop_start_exclusive_s=a,sop_end_inclusive_s=b,\n                           threshold=policy.threshold,score=float(scores[i]),outcome=outcome,\n                           future_annotation_complete=known,origin_evaluable=bool(evaluable[i]),\n                           early_seizure=bool(((e.onset_s>x)&(e.onset_s<=a)).any()),\n                           target_ids=target_ids,non_target_ids=non_target_ids))\n        next_allowed=b  # 35 minutes, including SPH, independent of observed outcomes.\n    alarm_columns=[\'alarm_id\',\'alarm_s\',\'sop_start_exclusive_s\',\'sop_end_inclusive_s\',\'threshold\',\'score\',\'outcome\',\n                   \'future_annotation_complete\',\'origin_evaluable\',\'early_seizure\',\'target_ids\',\'non_target_ids\']\n    alarm_table=pd.DataFrame(alarms,columns=alarm_columns)\n    targets=[]\n    for row in e[e.eligible_target.astype(bool)].itertuples():\n        opportunities=ready&(t>=row.onset_s-H)&(t<row.onset_s-policy.sph_s)\n        matches=[a for a in alarms if row.event_id in a[\'target_ids\']]\n        targets.append(dict(event_id=row.event_id,onset_s=float(row.onset_s),retained=True,\n                            forecast_opportunities=int(opportunities.sum()),has_opportunity=bool(opportunities.any()),\n                            predicted=bool(matches),matched_alarm_ids=[a[\'alarm_id\'] for a in matches],\n                            lead_time_s=float(row.onset_s-matches[0][\'alarm_s\']) if matches else np.nan))\n    target_table=pd.DataFrame(targets,columns=[\'event_id\',\'onset_s\',\'retained\',\'forecast_opportunities\',\'has_opportunity\',\'predicted\',\'matched_alarm_ids\',\'lead_time_s\'])\n    # Decision-bin burden: warning is active at decision t when alarm <= t < alarm+H.\n    changes=np.zeros(len(t)+1,dtype=np.int64)\n    for a in alarms:\n        changes[np.searchsorted(t,a[\'alarm_s\'],side=\'left\')]+=1\n        changes[np.searchsorted(t,a[\'sop_end_inclusive_s\'],side=\'left\')]-=1\n    active=np.cumsum(changes[:-1])>0\n    hits=sum(x[\'predicted\'] for x in targets);opp=sum(x[\'has_opportunity\'] for x in targets)\n    opp_hits=sum(x[\'predicted\'] and x[\'has_opportunity\'] for x in targets)\n    false=sum(a[\'outcome\']==\'false_alarm\' for a in alarms)\n    nt=sum(a[\'outcome\']==\'non_target_seizure\' for a in alarms)\n    eval_nt=sum(a[\'outcome\']==\'non_target_seizure\' and a[\'origin_evaluable\'] for a in alarms)\n    ready_h=float(ready.sum()*policy.decision_bin_s/3600)\n    eval_h=float(evaluable.sum()*policy.decision_bin_s/3600)\n    summary=dict(targets=len(targets),predicted_targets=hits,targets_with_opportunity=opp,\n                 sensitivity=hits/len(targets) if targets else None,\n                 opportunity_conditioned_sensitivity=opp_hits/opp if opp else None,\n                 recorded_decision_hours=len(t)*policy.decision_bin_s/3600,\n                 prediction_ready_hours=ready_h,evaluable_decision_hours=eval_h,\n                 prediction_availability=float(ready.mean()) if len(t) else None,\n                 alarm_count=len(alarms),false_alarms=false,non_target_seizure_alarms=nt,\n                 unscorable_alarms=sum(a[\'outcome\']==\'unscorable\' for a in alarms),\n                 incomplete_future_alarms=sum(not a[\'future_annotation_complete\'] for a in alarms),\n                 false_alarms_per_evaluable_hour=false/eval_h if eval_h else None,\n                 non_target_alarms_at_evaluable_origins=eval_nt,\n                 alarms_without_target_hit_per_evaluable_hour=(false+eval_nt)/eval_h if eval_h else None,\n                 alarms_per_prediction_ready_hour=len(alarms)/ready_h if ready_h else None,\n                 warning_evaluable_bins=int((active&evaluable).sum()),evaluable_bins=int(evaluable.sum()),\n                 warning_ready_bins=int((active&ready).sum()),ready_bins=int(ready.sum()),\n                 time_in_warning_evaluable=float((active&evaluable).sum()/evaluable.sum()) if evaluable.any() else None,\n                 time_in_warning_ready=float((active&ready).sum()/ready.sum()) if ready.any() else None)\n    return summary,alarm_table,target_table\n\n\ndef score_cohort(predictions,events,coverage_by_case,policy=ScoringPolicy()):\n    if \'case\' not in predictions or \'case\' not in events:raise ValueError(\'Case IDs required\')\n    unknown=(set(predictions[\'case\'])|set(events[\'case\']))-set(coverage_by_case)\n    if unknown:raise ValueError(f\'Missing coverage: {unknown}\')\n    summaries=[];alarms=[];targets=[]\n    for case,coverage in sorted(coverage_by_case.items()):\n        s,a,e=score_case(predictions[predictions[\'case\']==case],events[events[\'case\']==case],coverage,policy)\n        summaries.append(dict(case=case,**s));a.insert(0,\'case\',case);e.insert(0,\'case\',case)\n        alarms.append(a);targets.append(e)\n    table=pd.DataFrame(summaries)\n    def total(name):return float(table[name].sum())\n    def ratio(a,b):return total(a)/total(b) if total(b) else None\n    eligible=table[table.targets>0]\n    pooled=dict(targets=int(total(\'targets\')),predicted_targets=int(total(\'predicted_targets\')),\n                sensitivity=ratio(\'predicted_targets\',\'targets\'),\n                macro_case_sensitivity=float(eligible.sensitivity.mean()) if len(eligible) else None,\n                false_alarms_per_evaluable_hour=ratio(\'false_alarms\',\'evaluable_decision_hours\'),\n                time_in_warning_evaluable=ratio(\'warning_evaluable_bins\',\'evaluable_bins\'),\n                prediction_availability=ratio(\'prediction_ready_hours\',\'recorded_decision_hours\'),\n                unscorable_alarms=int(total(\'unscorable_alarms\')),non_target_seizure_alarms=int(total(\'non_target_seizure_alarms\')))\n    return pooled,table,pd.DataFrame([r for a in alarms for r in a.to_dict(\'records\')],columns=alarms[0].columns),pd.DataFrame([r for a in targets for r in a.to_dict(\'records\')],columns=targets[0].columns)\n\n\ndef validate_full_prediction_export(data, role, predictions, provenance):\n    """Use with FoldData from the loader notebook, before score_cohort.\n    Reject incomplete previews and mismatched roles/scalers. No signal reads.\n    """\n    import hashlib\n    if provenance.get(\'purpose\')!=\'full_chronological_predictions\' or provenance.get(\'smoke_only\',True):\n        raise ValueError(\'Smoke-test previews cannot be scored as full predictions\')\n    expected={\'outer\':data.outer,\'scope\':data.scope,\'role\':role,\n              \'scaler_sha256\':hashlib.sha256(data.scaler_path.read_bytes()).hexdigest(),\n              \'folds_sha256\':hashlib.sha256((data.run/\'locked_folds.json\').read_bytes()).hexdigest()}\n    for key,value in expected.items():\n        if provenance.get(key)!=value:raise ValueError(\'Prediction provenance mismatch: \'+key)\n    if not isinstance(provenance.get(\'model_checkpoint_sha256\'),str) or len(provenance[\'model_checkpoint_sha256\'])!=64:\n        raise ValueError(\'Model checkpoint hash required\')\n    required={\'relative_path\',\'index\',\'case\',\'decision_s\',\'prediction_ready\',\'score\'}\n    if not required.issubset(predictions):raise ValueError(\'Incomplete prediction schema\')\n    p=predictions.copy()\n    if p[[\'relative_path\',\'index\']].duplicated().any():raise ValueError(\'Duplicate prediction keys\')\n    rows=data.records(role);expected_paths={r[\'relative_path\'] for r in rows}\n    if set(p.relative_path)!=expected_paths:raise ValueError(\'Missing or unexpected recordings\')\n    for row in rows:\n        m=data.metadata(row);part=p[p.relative_path==row[\'relative_path\']].sort_values(\'index\')\n        n=len(m[\'decision_s\'])\n        if len(part)!=n or not np.array_equal(part[\'index\'].to_numpy(),np.arange(n)):\n            raise ValueError(\'Missing windows: \'+row[\'relative_path\'])\n        if not (part[\'case\']==row[\'case\']).all() or not np.allclose(part.decision_s,m[\'decision_s\'],rtol=0,atol=1e-6):\n            raise ValueError(\'Case/timestamps mismatch\')\n        if not part.prediction_ready.isin([True,False]).all() or not np.array_equal(part.prediction_ready.to_numpy(dtype=bool),m[\'prediction_ready\']):\n            raise ValueError(\'Prediction availability differs from locked loader policy\')\n    # Include annotated recordings with unusable montage in outcome coverage.\n    # Prediction availability and annotation availability are distinct concepts.\n    allowed=set(data.roles[role])\n    allrows=[r for r in data.inventory if r[\'subject\'] in allowed and not r[\'excluded_case\']]\n    cases={r[\'case\'] for r in allrows}\n    coverage={case:[(r[\'start_s\'],r[\'end_s\']) for r in allrows if r[\'case\']==case] for case in cases}\n    events=pd.DataFrame([e for e in data.events if e[\'case\'] in cases])\n    return p,events,coverage\n'
exec(compile(SOURCE_3, "embedded_3.py", "exec"))

In [ ]:
SOURCE_4 = 'import os, json, hashlib, shutil, tempfile, time\nfrom pathlib import Path\nfrom collections import defaultdict\nimport numpy as np\nimport pandas as pd\nimport h5py\nimport torch\nfrom torch.nn import functional as F\n\nVARIANTS=(\'classifier\',\'koopman\',\'physics\')\n\ndef digest(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for block in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef json_atomic(path,value):\n    path=Path(path);tmp=path.with_name(path.name+\'.tmp\')\n    tmp.write_text(json.dumps(value,indent=2,allow_nan=False));os.replace(tmp,path)\n\ndef init_pilot(data,destination,settings,code_sha):\n    destination=Path(destination);destination.mkdir(parents=True,exist_ok=True)\n    plan=dict(stage=\'matched_pilot_v1\',outer=data.outer,scope=data.scope,roles=data.roles,\n              settings=settings,code_sha256=code_sha,scaler_sha256=digest(data.scaler_path),\n              folds_sha256=digest(data.run/\'locked_folds.json\'),inventory_sha256=digest(data.run/\'locked_inventory.json\'),\n              variants=list(VARIANTS),threshold_selected=False)\n    if data.scope==\'outer_refit\':raise ValueError(\'Pilot restricted to development inner folds\')\n    p=destination/\'pilot_plan.json\'\n    if p.exists():\n        if json.loads(p.read_text())!=plan:raise ValueError(\'Pilot plan changed. Use a fresh pilot directory.\')\n    else:json_atomic(p,plan)\n    return hashlib.sha256(json.dumps(plan,sort_keys=True).encode()).hexdigest()\n\ndef build_bank(data,destination,settings,fingerprint,local_root):\n    destination=Path(destination);local_root=Path(local_root);local_root.mkdir(parents=True,exist_ok=True)\n    saved=destination/\'training_bank.h5\';manifest=destination/\'training_samples.json\'\n    if not saved.exists():\n        candidates=data.index(\'train\',\'pairs\');rng=np.random.default_rng(settings[\'seed\'])\n        chosen=[];eligible_counts={}\n        for label in [0,1,2]:\n            group=[(r,i) for r,i in candidates if int(data.metadata(r)[\'label\'][i])==label]\n            eligible_counts[str(label)]=len(group)\n            if not group:raise ValueError(f\'No eligible training pairs for class {label}\')\n            pick=rng.choice(len(group),min(settings[\'pool_per_class\'],len(group)),replace=False)\n            chosen.extend((group[int(j)][0],group[int(j)][1],label) for j in pick)\n        sample_manifest=[dict(relative_path=r[\'relative_path\'],subject=r[\'subject\'],index=i,label=k) for r,i,k in chosen]\n        groups=defaultdict(list)\n        for j,(r,i,k) in enumerate(chosen):groups[r[\'relative_path\']].append((j,r,i))\n        tmp=local_root/\'bank_build.h5\';C=len(data.cfg[\'channels\']);L=int(data.cfg[\'window_s\']*data.cfg[\'fs\']);N=len(chosen)\n        with h5py.File(tmp,\'w\') as out:\n            out.attrs[\'fingerprint\']=fingerprint\n            for key,shape in [(\'raw\',(N,C,L)),(\'next_raw\',(N,C,L)),(\'spec\',(N,C,5)),(\'next_spec\',(N,C,5))]:\n                out.create_dataset(key,shape,dtype=\'f4\')\n            out[\'labels\']=np.array([k for _,_,k in chosen],dtype=np.int64)\n            for fi,(relative,entries) in enumerate(sorted(groups.items())):\n                row=entries[0][1];needed=sorted({idx for _,_,i in entries for idx in [i,i+1]})\n                lookup={idx:j for j,idx in enumerate(needed)}\n                with h5py.File(data.path(row),\'r\') as h:\n                    raw=h[\'raw_uv\'][needed];spec=h[\'features\'][needed]\n                s=data.scaler\n                raw=((raw-s[\'raw_mean\'][:,None])/s[\'raw_std\'][:,None]).astype(\'f4\')\n                spec=((spec-s[\'feature_mean\'])/s[\'feature_std\']).astype(\'f4\')\n                if not np.isfinite(raw).all() or not np.isfinite(spec).all():raise ValueError(\'Nonfinite bank inputs\')\n                for j,_,i in entries:\n                    out[\'raw\'][j]=raw[lookup[i]];out[\'next_raw\'][j]=raw[lookup[i+1]]\n                    out[\'spec\'][j]=spec[lookup[i]];out[\'next_spec\'][j]=spec[lookup[i+1]]\n                if fi%10==0 or fi+1==len(groups):print(f\'Bank files: {fi+1}/{len(groups)}\',flush=True)\n        json_atomic(manifest,dict(fingerprint=fingerprint,eligible_training_pairs=eligible_counts,samples=sample_manifest))\n        pending=saved.with_suffix(\'.h5.tmp\');shutil.copyfile(tmp,pending);os.replace(pending,saved)\n        json_atomic(destination/\'bank_integrity.json\',dict(sha256=digest(saved),fingerprint=fingerprint))\n        tmp.unlink()\n    if not manifest.exists():raise ValueError(\'Training bank manifest missing\')\n    info=json.loads(manifest.read_text())\n    if info[\'fingerprint\']!=fingerprint:raise ValueError(\'Wrong training bank\')\n    local=local_root/\'training_bank.h5\'\n    integrity_path=destination/\'bank_integrity.json\'\n    # Recover an interruption after the atomic bank copy but before its hash was saved.\n    if not integrity_path.exists():json_atomic(integrity_path,dict(sha256=digest(saved),fingerprint=fingerprint))\n    integrity=json.loads(integrity_path.read_text())\n    if integrity[\'fingerprint\']!=fingerprint:raise ValueError(\'Bank integrity mismatch\')\n    if not local.exists() or digest(local)!=integrity[\'sha256\']:\n        shutil.copyfile(saved,local)\n    if digest(local)!=integrity[\'sha256\']:raise ValueError(\'Training bank checksum mismatch\')\n    with h5py.File(local,\'r\') as h:\n        if h.attrs[\'fingerprint\']!=fingerprint:raise ValueError(\'Wrong bank attributes\')\n        bank={k:h[k][:] for k in [\'raw\',\'spec\',\'next_raw\',\'next_spec\',\'labels\']}\n    print(\'Loaded bank:\',len(bank[\'labels\']),\'pairs;\',round(sum(v.nbytes for v in bank.values())/2**20),\'MiB\')\n    return bank,info\n\ndef batch_indices(bank,step,settings):\n    rng=np.random.default_rng(settings[\'seed\']+100000+step)\n    ids=np.concatenate([rng.choice(np.flatnonzero(bank[\'labels\']==k),settings[\'batch_per_class\'],replace=True) for k in [0,1,2]])\n    rng.shuffle(ids);return ids\n\ndef pilot_batch(bank,ids,device):\n    return {k:torch.as_tensor(v[ids],device=device,dtype=torch.long if k==\'labels\' else torch.float32) for k,v in bank.items()}\n\ndef new_variant(variant,channels,seed,device):\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)\n    model=(PhysicsMVModel(channels=channels) if variant==\'physics\' else CoreMVModel(channels=channels)).to(device)\n    if variant==\'classifier\':model.U.requires_grad_(False);model.V.requires_grad_(False)\n    return model\n\ndef variant_loss(model,batch,variant):\n    if variant==\'physics\':return physics_loss(model,batch)\n    if variant==\'koopman\':return core_loss(model,batch)\n    logits,z=model(batch[\'raw\'],batch[\'spec\']);ce=F.cross_entropy(logits,batch[\'labels\'])\n    return ce,dict(classification=ce),z\n\n@torch.no_grad()\ndef pilot_diagnostics(model,batch,variant):\n    if variant==\'physics\':return physics_diagnostics(model,batch)\n    z=model.encode(batch[\'raw\'],batch[\'spec\']);center=z.double()-z.double().mean(0)\n    eig=torch.linalg.svdvals(center).square();p=eig/eig.sum().clamp_min(1e-30);p=p[p>0]\n    d=dict(latent_mean_std=float(z.std(0,unbiased=False).mean().cpu()),\n           latent_effective_rank=float(torch.exp(-(p*p.log()).sum()).cpu()) if len(p) else 0.,\n           effective_rank_upper_bound=min(len(z)-1,z.shape[-1]))\n    if variant!=\'classifier\':\n        K=model.operator_matrix().double().cpu()\n        d.update(koopman_spectral_radius=float(torch.linalg.eigvals(K).abs().max()),\n                 koopman_largest_singular_value=float(torch.linalg.svdvals(K).max()))\n    return d\n\ndef train_variant(variant,bank,settings,destination,fingerprint,device,stop_after=None):\n    folder=Path(destination)/variant;folder.mkdir(exist_ok=True)\n    checkpoint=folder/\'last.pt\'\n    model=new_variant(variant,bank[\'raw\'].shape[1],settings[\'seed\'],device)\n    optimizer=torch.optim.Adam([p for p in model.parameters() if p.requires_grad],lr=settings[\'learning_rate\'])\n    start=0;logs=[];diagnostics=[]\n    if checkpoint.exists():\n        state=torch.load(checkpoint,map_location=device,weights_only=False)  # Only this pilot\'s own checkpoint.\n        if state[\'fingerprint\']!=fingerprint or state[\'variant\']!=variant:raise ValueError(\'Checkpoint provenance mismatch\')\n        model.load_state_dict(state[\'model\']);optimizer.load_state_dict(state[\'optimizer\'])\n        start=state[\'step\'];logs=state[\'logs\'];diagnostics=state[\'diagnostics\']\n        print(variant,\'resuming at step\',start,flush=True)\n    limit=min(settings[\'steps\'],stop_after) if stop_after is not None else settings[\'steps\']\n    model.train()\n    for step in range(start,limit):\n        batch=pilot_batch(bank,batch_indices(bank,step,settings),device)\n        optimizer.zero_grad(set_to_none=True);loss,parts,z=variant_loss(model,batch,variant)\n        if not torch.isfinite(loss):raise RuntimeError(f\'{variant}: nonfinite loss at step {step+1}\')\n        record=dict(step=step+1,total=float(loss.detach().cpu()),**{k:float(v.detach().cpu()) for k,v in parts.items()})\n        checkpoint_due=(step+1)%settings[\'checkpoint_every\']==0 or step+1==limit\n        if variant==\'physics\' and checkpoint_due:\n            w=model.raw_project.weight\n            gc=torch.autograd.grad(parts[\'classification\'],w,retain_graph=True)[0]\n            gp=torch.autograd.grad(.2*parts[\'physics\'],w,retain_graph=True)[0]\n            record[\'weighted_physics_to_CE_gradient_ratio\']=float((gp.norm()/gc.norm().clamp_min(1e-12)).detach().cpu())\n        loss.backward()\n        for name,p in model.named_parameters():\n            if p.requires_grad and (p.grad is None or not torch.isfinite(p.grad).all()):\n                raise RuntimeError(f\'{variant}: missing/nonfinite gradient at {name}\')\n        grad=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],5.)\n        optimizer.step()\n        if not all(torch.isfinite(p).all() for p in model.parameters()):raise RuntimeError(\'Nonfinite model parameter\')\n        record[\'preclip_gradient_norm\']=float(grad.detach().cpu());logs.append(record)\n        if checkpoint_due:\n            diag=pilot_diagnostics(model,batch,variant)\n            if diag[\'latent_mean_std\']<=1e-8:raise RuntimeError(\'Latent variation collapsed on diagnostic batch\')\n            if variant==\'physics\' and diag[\'max_difference_16_vs_256\']>=.005:\n                raise RuntimeError(\'Physics integration error exceeds 0.005; review before resuming\')\n            diagnostics.append(dict(step=step+1,**diag))\n            state=dict(fingerprint=fingerprint,variant=variant,step=step+1,model=model.state_dict(),\n                       optimizer=optimizer.state_dict(),logs=logs,diagnostics=diagnostics)\n            tmp=checkpoint.with_suffix(\'.pt.tmp\');torch.save(state,tmp);os.replace(tmp,checkpoint)\n            pd.DataFrame(logs).to_csv(folder/\'training_log.csv\',index=False)\n            json_atomic(folder/\'training_diagnostics.json\',diagnostics)\n            print(variant,\'step\',step+1,\'/\',settings[\'steps\'],\'loss\',round(record[\'total\'],5),flush=True)\n    return model,logs,diagnostics\n\n@torch.no_grad()\ndef infer_file(models,data,row,device,chunk=128):\n    m=data.metadata(row);N=len(m[\'label\']);prob=np.full((len(models),N,3),np.nan,dtype=\'f4\')\n    for model in models:model.eval()\n    with h5py.File(data.path(row),\'r\') as h:\n        for first in range(0,N,chunk):\n            end=min(first+chunk,N);valid=m[\'valid_signal\'][first:end]\n            if not valid.any():continue\n            # Read sequentially once, then use the same input for every variant.\n            raw=h[\'raw_uv\'][first:end][valid];spec=h[\'features\'][first:end][valid];s=data.scaler\n            raw=((raw-s[\'raw_mean\'][:,None])/s[\'raw_std\'][:,None]).astype(\'f4\')\n            spec=((spec-s[\'feature_mean\'])/s[\'feature_std\']).astype(\'f4\')\n            if not np.isfinite(raw).all() or not np.isfinite(spec).all():raise ValueError(\'Nonfinite inference inputs\')\n            x=torch.as_tensor(raw,device=device);f=torch.as_tensor(spec,device=device)\n            ids=np.flatnonzero(valid)+first\n            for j,model in enumerate(models):\n                logits,_=model(x,f);p=logits.softmax(-1).cpu().numpy()\n                if not np.isfinite(p).all():raise ValueError(\'Nonfinite predictions\')\n                prob[j,ids]=p\n    H=data.history;scores=np.full((len(models),N),np.nan,dtype=\'f4\')\n    if H<8:raise ValueError(\'History shorter than Top-8\')\n    if N>=H:\n        for j in range(len(models)):\n            windows=np.lib.stride_tricks.sliding_window_view(prob[j,:,1],H)\n            fused=np.partition(windows,H-8,axis=1)[:,-8:].mean(axis=1)\n            scores[j,H-1:]=fused\n            scores[j,~m[\'prediction_ready\']]=np.nan\n            if not np.isfinite(scores[j,m[\'prediction_ready\']]).all():raise ValueError(\'Missing ready scores\')\n    return prob,scores\n\ndef validation_export(models,data,destination,fingerprint,device,settings):\n    destination=Path(destination);folder=destination/\'validation_cache\';folder.mkdir(exist_ok=True)\n    hashes={v:digest(destination/v/\'last.pt\') for v in VARIANTS}\n    for v in VARIANTS:\n        state=torch.load(destination/v/\'last.pt\',map_location=\'cpu\',weights_only=False)\n        if state[\'step\']!=settings[\'steps\'] or state[\'fingerprint\']!=fingerprint:raise ValueError(\'Training is not complete\')\n    pred_id=hashlib.sha256(json.dumps(dict(pilot=fingerprint,checkpoints=hashes),sort_keys=True).encode()).hexdigest()\n    records=data.records(\'validation\');frames={v:[] for v in VARIANTS}\n    for k,row in enumerate(records):\n        path=folder/(row[\'case\']+\'__\'+Path(row[\'relative_path\']).stem+\'.npz\')\n        m=data.metadata(row);cached=False\n        if path.exists():\n            with np.load(path,allow_pickle=False) as z:\n                if str(z[\'fingerprint\'])==pred_id and z[\'scores\'].shape==(3,len(m[\'label\'])):\n                    prob=z[\'probabilities\'];scores=z[\'scores\'];cached=True\n        if not cached:\n            prob,scores=infer_file(models,data,row,device,settings[\'inference_chunk\'])\n            tmp=path.with_suffix(\'.npz.tmp\')\n            with tmp.open(\'wb\') as f:np.savez_compressed(f,fingerprint=pred_id,probabilities=prob,scores=scores)\n            os.replace(tmp,path)\n        for j,v in enumerate(VARIANTS):\n            f=pd.DataFrame(dict(relative_path=row[\'relative_path\'],index=np.arange(len(m[\'label\'])),case=row[\'case\'],\n                                decision_s=m[\'decision_s\'],prediction_ready=m[\'prediction_ready\'],score=scores[j],\n                                p_interictal=prob[j,:,0],p_preictal=prob[j,:,1],p_ictal=prob[j,:,2]))\n            frames[v].append(f)\n        print(f\'Validation files: {k+1}/{len(records)}\'+(\' (cached)\' if cached else \'\'),flush=True)\n    exports={}\n    for v in VARIANTS:\n        p=pd.concat(frames[v],ignore_index=True)\n        provenance=dict(purpose=\'full_chronological_predictions\',smoke_only=False,outer=data.outer,scope=data.scope,\n                        role=\'validation\',scaler_sha256=digest(data.scaler_path),folds_sha256=digest(data.run/\'locked_folds.json\'),\n                        model_checkpoint_sha256=hashes[v],pilot_fingerprint=fingerprint,variant=v)\n        validate_full_prediction_export(data,\'validation\',p,provenance)\n        p.to_csv(destination/v/\'validation_predictions.csv\',index=False)\n        json_atomic(destination/v/\'prediction_provenance.json\',provenance)\n        exports[v]=(p,provenance)\n    return exports\n\ndef window_metrics(data,predictions):\n    """Descriptive classification on ready labelled windows; no class balancing."""\n    confusion=np.zeros((3,3),dtype=np.int64);loss_sum=0.;total=0\n    for row in data.records(\'validation\'):\n        m=data.metadata(row);p=predictions[predictions.relative_path==row[\'relative_path\']].sort_values(\'index\')\n        probs=p[[\'p_interictal\',\'p_preictal\',\'p_ictal\']].to_numpy()\n        keep=m[\'prediction_ready\']&(m[\'label\']>=0)\n        y=m[\'label\'][keep].astype(int);pr=probs[keep]\n        if len(y):\n            if not np.isfinite(pr).all():raise ValueError(\'Nonfinite scored probabilities\')\n            np.add.at(confusion,(y,pr.argmax(1)),1)\n            loss_sum+=float(-np.log(np.maximum(pr[np.arange(len(y)),y],1e-12)).sum());total+=len(y)\n    recall=np.divide(confusion.diagonal(),confusion.sum(1),out=np.full(3,np.nan),where=confusion.sum(1)>0)\n    return dict(ready_labelled_windows=total,cross_entropy=loss_sum/total if total else None,\n                balanced_accuracy=float(np.nanmean(recall)) if total else None,\n                class_recalls=[float(x) if np.isfinite(x) else None for x in recall],confusion_matrix=confusion.tolist())\n\ndef score_pilot(data,exports,destination,settings):\n    rows=[];window={}\n    for v,(p,provenance) in exports.items():\n        q,e,coverage=validate_full_prediction_export(data,\'validation\',p,provenance)\n        window[v]=window_metrics(data,p)\n        for threshold in settings[\'threshold_grid\']:\n            pooled,cases,alarms,targets=score_cohort(q,e,coverage,ScoringPolicy(threshold=threshold))\n            rows.append(dict(variant=v,threshold=threshold,**pooled,\n                             evaluable_decision_hours=float(cases.evaluable_decision_hours.sum()),\n                             prediction_ready_hours=float(cases.prediction_ready_hours.sum()),\n                             total_alarms=int(cases.alarm_count.sum()),false_alarms=int(cases.false_alarms.sum()),\n                             targets_with_opportunity=int(cases.targets_with_opportunity.sum()),\n                             incomplete_future_alarms=int(cases.incomplete_future_alarms.sum())))\n            # Save case-level results and target hits at every predeclared threshold.\n            dest=Path(destination)/v/\'threshold_reports\';dest.mkdir(exist_ok=True)\n            tag=f\'{threshold:.2f}\'\n            cases.to_csv(dest/f\'cases_{tag}.csv\',index=False)\n            targets.to_json(dest/f\'targets_{tag}.json\',orient=\'records\',indent=2)\n            alarms.to_json(dest/f\'alarms_{tag}.json\',orient=\'records\',indent=2)\n        print(\'Scoring complete:\',v,flush=True)\n    table=pd.DataFrame(rows);table.to_csv(Path(destination)/\'validation_threshold_curves.csv\',index=False)\n    json_atomic(Path(destination)/\'window_metrics.json\',window)\n    return table,window\n'
exec(compile(SOURCE_4, "embedded_4.py", "exec"))

In [ ]:
FULL_SOURCE="\nclass LazyArray:\n    def __init__(self,bank,key,shape): self.bank,self.key,self.shape=bank,key,shape\n    def __getitem__(self,ids): return self.bank.load(ids)[self.key]\nclass FullBank(dict):\n    def __init__(self,data):\n        self.data=data; self.samples=data.index('train','pairs'); self.cached_ids=None\n        labels=np.array([int(data.metadata(r)['label'][i]) for r,i in self.samples],dtype=np.int64)\n        if any(not (labels==k).any() for k in range(3)): raise ValueError('Training class absent')\n        C=len(data.cfg['channels']);L=int(data.cfg['fs']*data.cfg['window_s'])\n        super().__init__(labels=labels)\n        for k,shape in [('raw',(len(labels),C,L)),('next_raw',(len(labels),C,L)),('spec',(len(labels),C,5)),('next_spec',(len(labels),C,5))]: self[k]=LazyArray(self,k,shape)\n        self.epoch=-1; self.order=None\n    def load(self,ids):\n        ids=np.asarray(ids)\n        if self.cached_ids is not None and np.array_equal(ids,self.cached_ids): return self.cached\n        out={k:np.empty((len(ids),)+self[k].shape[1:],dtype='f4') for k in ['raw','next_raw','spec','next_spec']}\n        groups=defaultdict(list)\n        for j,idx in enumerate(ids):\n            row,i=self.samples[int(idx)];groups[str(self.data.path(row))].append((j,i))\n        s=self.data.scaler\n        for path,entries in groups.items():\n            with h5py.File(path,'r') as h:\n                for j,i in entries:\n                    for key,offset in [('raw',0),('next_raw',1)]:out[key][j]=(h['raw_uv'][i+offset]-s['raw_mean'][:,None])/s['raw_std'][:,None]\n                    for key,offset in [('spec',0),('next_spec',1)]:out[key][j]=(h['features'][i+offset]-s['feature_mean'])/s['feature_std']\n        if any(not np.isfinite(x).all() for x in out.values()):raise ValueError('Nonfinite training input')\n        self.cached_ids=ids.copy();self.cached=out;return out\n\ndef batch_indices(bank,step,settings):\n    per_epoch=settings['steps_per_epoch'];epoch,offset=divmod(step,per_epoch)\n    if bank.epoch!=epoch:\n        rng=np.random.default_rng(settings['seed']+100000+epoch);groups=[]\n        largest=max(int((bank['labels']==k).sum()) for k in range(3))\n        for k in range(3):\n            ids=np.flatnonzero(bank['labels']==k); parts=[];remaining=largest\n            while remaining:\n                p=rng.permutation(ids)[:remaining];parts.append(p);remaining-=len(p)\n            groups.append(np.concatenate(parts))\n        order=np.stack(groups,axis=1).reshape(-1)\n        # Each group of three contains one member of every class.\n        bank.order=order;bank.epoch=epoch\n    batch=settings['batch_size'];return bank.order[offset*batch:min((offset+1)*batch,len(bank.order))]\n\ndef fit_full(data,dest,seed):\n    dest=Path(dest);dest.mkdir(parents=True,exist_ok=True)\n    bank=FullBank(data);largest=max(int((bank['labels']==k).sum()) for k in range(3))\n    settings=dict(SETTINGS,seed=seed,steps_per_epoch=int(np.ceil(3*largest/SETTINGS['batch_size'])))\n    settings['steps']=settings['epochs']*settings['steps_per_epoch']\n    plan=dict(implementation=IMPLEMENTATION_SHA,settings=settings,roles=data.roles,outer=data.outer,scope=data.scope,\n              scaler=digest(data.scaler_path),folds=digest(data.run/'locked_folds.json'),inventory=digest(data.run/'locked_inventory.json'),events=digest(data.run/'locked_events.json'),config=digest(data.run/'config.json'))\n    fingerprint=hashlib.sha256(json.dumps(plan,sort_keys=True).encode()).hexdigest()\n    p=dest/'fit_plan.json'\n    if p.exists() and json.loads(p.read_text())!=plan:raise ValueError('Plan differs; use a new experiment directory')\n    json_atomic(p,plan)\n    print(data.outer,data.scope,'pairs',len(bank['labels']),'updates',settings['steps'],flush=True)\n    models=[train_variant(v,bank,settings,dest,fingerprint,DEVICE)[0] for v in VARIANTS]\n    return models,settings,fingerprint\n\ndef select_threshold(reports):\n    rows=[]\n    for threshold,frames in reports.items():\n        c=pd.concat(frames,ignore_index=True)\n        if c['case'].duplicated().any():raise ValueError('Validation cases overlap across inner folds')\n        eligible=c[c.targets>0];hours=float(c.evaluable_decision_hours.sum());bins=float(c.evaluable_bins.sum())\n        if not len(eligible) or hours<=0 or bins<=0:raise ValueError('Insufficient validation targets or coverage')\n        rows.append(dict(threshold=float(threshold),macro_sensitivity=float(eligible.sensitivity.mean()),\n                         pooled_sensitivity=float(c.predicted_targets.sum()/c.targets.sum()),\n                         warning=float(c.warning_evaluable_bins.sum()/bins),false_alarm_rate=float(c.false_alarms.sum()/hours)))\n    table=pd.DataFrame(rows)\n    valid=table[table.warning<=SETTINGS['warning_cap']+1e-12]\n    if valid.empty:raise ValueError('No feasible threshold, including no-alert fallback')\n    best=valid.sort_values(['macro_sensitivity','false_alarm_rate','warning','threshold'],ascending=[False,True,True,False]).iloc[0].to_dict()\n    best['status']='no_validation_targets_predicted' if best['macro_sensitivity']==0 else 'selected'\n    return best,table\n\ndef run_outer(outer,seed):\n    root=EXPERIMENT/outer/str(seed);root.mkdir(parents=True,exist_ok=True)\n    fold=next(f for f in FOLDS if f['id']==outer)\n    reports={v:{t:[] for t in SETTINGS['threshold_grid']} for v in VARIANTS}\n    for inner in fold['inner_folds']:\n        data=FoldData(RUN,outer,inner['id']);dest=root/inner['id']\n        models,settings,fp=fit_full(data,dest,seed)\n        exports=validation_export(models,data,dest,fp,DEVICE,settings)\n        for v,(p,prov) in exports.items():\n            q,e,cov=validate_full_prediction_export(data,'validation',p,prov)\n            for t in settings['threshold_grid']:\n                pooled,cases,alarms,targets=score_cohort(q,e,cov,ScoringPolicy(threshold=t))\n                reports[v][t].append(cases)\n                cases.to_csv(dest/v/f'cases_{t:.2f}.csv',index=False)\n        del models,exports,data\n        if torch.cuda.is_available():torch.cuda.empty_cache()\n    selections={}\n    for v in VARIANTS:\n        best,table=select_threshold(reports[v]);selections[v]=best\n        table.to_csv(root/f'{v}_inner_thresholds.csv',index=False)\n    selection_path=root/'locked_selection.json'\n    if selection_path.exists() and json.loads(selection_path.read_text())!=selections:raise ValueError('Locked selection changed')\n    json_atomic(selection_path,selections)\n    print('Locked validation selections:',selections,flush=True)\n    data=FoldData(RUN,outer,'outer_refit');dest=root/'outer_refit'\n    models,settings,fp=fit_full(data,dest,seed)\n    exports=test_export(models,data,dest,fp,DEVICE,settings)\n    results=[]\n    for v,(p,prov) in exports.items():\n        q,e,cov=validate_full_prediction_export(data,'test',p,prov)\n        pooled,cases,alarms,targets=score_cohort(q,e,cov,ScoringPolicy(threshold=selections[v]['threshold']))\n        cases.to_csv(dest/v/'test_cases.csv',index=False)\n        alarms.to_csv(dest/v/'test_alarms.csv',index=False);targets.to_csv(dest/v/'test_targets.csv',index=False)\n        results.append(dict(outer=outer,seed=seed,variant=v,threshold=selections[v]['threshold'],**pooled))\n    pd.DataFrame(results).to_csv(root/'outer_results.csv',index=False)\n    json_atomic(root/'status.json',dict(status='outer_complete',selection_sha256=digest(selection_path)))\n    return results\n"
exec(compile(FULL_SOURCE,"full_run.py","exec"))

In [ ]:
TEST_EXPORT_SOURCE="def test_export(models,data,destination,fingerprint,device,settings):\n    destination=Path(destination);folder=destination/'test_cache';folder.mkdir(exist_ok=True)\n    hashes={v:digest(destination/v/'last.pt') for v in VARIANTS}\n    for v in VARIANTS:\n        state=torch.load(destination/v/'last.pt',map_location='cpu',weights_only=False)\n        if state['step']!=settings['steps'] or state['fingerprint']!=fingerprint:raise ValueError('Training is not complete')\n    pred_id=hashlib.sha256(json.dumps(dict(pilot=fingerprint,checkpoints=hashes),sort_keys=True).encode()).hexdigest()\n    records=data.records('test');frames={v:[] for v in VARIANTS}\n    for k,row in enumerate(records):\n        path=folder/(row['case']+'__'+Path(row['relative_path']).stem+'.npz')\n        m=data.metadata(row);cached=False\n        if path.exists():\n            with np.load(path,allow_pickle=False) as z:\n                if str(z['fingerprint'])==pred_id and z['scores'].shape==(3,len(m['label'])):\n                    prob=z['probabilities'];scores=z['scores'];cached=True\n        if not cached:\n            prob,scores=infer_file(models,data,row,device,settings['inference_chunk'])\n            tmp=path.with_suffix('.npz.tmp')\n            with tmp.open('wb') as f:np.savez_compressed(f,fingerprint=pred_id,probabilities=prob,scores=scores)\n            os.replace(tmp,path)\n        for j,v in enumerate(VARIANTS):\n            f=pd.DataFrame(dict(relative_path=row['relative_path'],index=np.arange(len(m['label'])),case=row['case'],\n                                decision_s=m['decision_s'],prediction_ready=m['prediction_ready'],score=scores[j],\n                                p_interictal=prob[j,:,0],p_preictal=prob[j,:,1],p_ictal=prob[j,:,2]))\n            frames[v].append(f)\n        print(f'Test files: {k+1}/{len(records)}'+(' (cached)' if cached else ''),flush=True)\n    exports={}\n    for v in VARIANTS:\n        p=pd.concat(frames[v],ignore_index=True)\n        provenance=dict(purpose='full_chronological_predictions',smoke_only=False,outer=data.outer,scope=data.scope,\n                        role='test',scaler_sha256=digest(data.scaler_path),folds_sha256=digest(data.run/'locked_folds.json'),\n                        model_checkpoint_sha256=hashes[v],pilot_fingerprint=fingerprint,variant=v)\n        validate_full_prediction_export(data,'test',p,provenance)\n        p.to_csv(destination/v/'test_predictions.csv',index=False)\n        json_atomic(destination/v/'prediction_provenance.json',provenance)\n        exports[v]=(p,provenance)\n    return exports\n\n"
exec(compile(TEST_EXPORT_SOURCE,"test_export.py","exec"))

## 3 · Set the fixed experiment plan
Leave RUN_EXPERIMENT=False for the first pass. Set it to True and rerun Section 4 to train. Keep EXPERIMENT unchanged to resume. Settings and source changes require a new experiment folder. Three epochs is a starting full-data budget; fix any revised budget using development data before outer testing.

In [ ]:
RUN=Path('/content/gdrive/MyDrive/mvPIKN_preprocessing/run_20261028T165011846018Z')
EXPERIMENT=RUN/'nested_full_v1_1'
OUTERS=['outer_chb01']  # None selects all locked outer folds.
SEEDS=[20261028]       # Full repeated study: [20261028,20261029,20261030]
RUN_EXPERIMENT=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark=False
SETTINGS=dict(epochs=3,batch_size=24,checkpoint_every=250,learning_rate=0.001,inference_chunk=128,
              warning_cap=0.20,threshold_grid=[round(float(x),2) for x in np.linspace(0,1,21)]+[1.01],
              runtime=dict(torch_version=str(torch.__version__),device=str(DEVICE),gpu_name=torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None))
IMPLEMENTATION_SHA=hashlib.sha256(('\n'.join([SOURCE_0,SOURCE_1,SOURCE_2,SOURCE_3,SOURCE_4,FULL_SOURCE,TEST_EXPORT_SOURCE])).encode()).hexdigest()
FOLDS=json.loads((RUN/'locked_folds.json').read_text())
chosen=[f['id'] for f in FOLDS] if OUTERS is None else OUTERS
assert set(chosen)<=set(f['id'] for f in FOLDS)
EXPERIMENT.mkdir(parents=True,exist_ok=True)
plan=dict(settings=SETTINGS,implementation=IMPLEMENTATION_SHA,folds=digest(RUN/'locked_folds.json'),seeds=SEEDS)
p=EXPERIMENT/'experiment_plan.json'
if p.exists() and json.loads(p.read_text())!=plan:raise ValueError('Experiment changed; choose a fresh EXPERIMENT path')
json_atomic(p,plan)
display(pd.DataFrame([dict(outer=o,seed=s,fits=4*len(VARIANTS)) for o in chosen for s in SEEDS]))
print('Device:',DEVICE,'; output:',EXPERIMENT)
print('Drive random I/O can make this run slow. Resume uses the same settings and runtime.')

## 3B · Faster training and checkpoint compatibility
Run this cell after Section 3, before Section 4. It installs an I/O-only loader update. Keep EXPERIMENT at nested_full_v1_1 and preserve your seed, GPU type and training settings to resume existing checkpoints.

On first use, each training file is read once from Drive into uncompressed NumPy arrays on Colab local disk. This staging can take time and substantial disk space. Cached files are reused while the runtime survives; after a runtime reset they are rebuilt. Drive checkpoints persist. Individual training batches then read local arrays in groups, with up to 16 files mapped at once. Cache identity includes source path, size and modification time. A small exact tensor comparison runs before training. Loader timing is printed every 250 batches. No training batch order, class balance, learning rate or epoch count is changed.

The original experiment fingerprint is deliberately preserved because this patch changes data access only; its separate source hash is recorded in io_extension.json. This cell is safe to rerun. When upgrading from the previous loader, restart the runtime once to discard the earlier wrappers. Validation replay still uses the existing sequential file reader.


In [ ]:
IO_SOURCE="# I/O-only extension: training plan, sampling, model and optimizer remain unchanged.\nimport time\nfrom collections import OrderedDict\nLOCAL_TRAINING_CACHE = Path('/content/chbmit_training_cache')\n# Store originals on the class once; global aliases can be overwritten on rerun.\nif '_io_original_init' not in FullBank.__dict__:\n    FullBank._io_original_init = FullBank.__init__\n    FullBank._io_original_load = FullBank.load\n\ndef _fast_bank_init(self, data):\n    FullBank._io_original_init(self, data)\n    self.local_files={}; self.mmaps=OrderedDict(); self.io_seconds=0.; self.io_batches=0\n    root=LOCAL_TRAINING_CACHE/hashlib.sha256(str(data.run).encode()).hexdigest()[:16]\n    root.mkdir(parents=True,exist_ok=True)\n    records={str(data.path(r)):r for r,i in self.samples}\n    for fi,(path,row) in enumerate(records.items()):\n        src=Path(path);st=src.stat()\n        identity=dict(path=path,size=st.st_size,mtime_ns=st.st_mtime_ns)\n        folder=root/hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:24]\n        folder.mkdir(exist_ok=True); marker=folder/'complete.json'\n        if not marker.exists():\n            start=time.perf_counter()\n            with h5py.File(src,'r') as h:\n                needed=sum(int(np.prod(h[k].shape))*h[k].dtype.itemsize for k in ['raw_uv','features'])\n                if shutil.disk_usage(root).free < needed+2*1024**3:\n                    raise RuntimeError('Local cache disk space insufficient. Free local space or use a larger runtime; Drive checkpoints are safe.')\n                for key in ['raw_uv','features']:\n                    ds=h[key];tmp=folder/(key+'.tmp.npy')\n                    a=np.lib.format.open_memmap(tmp,mode='w+',dtype=ds.dtype,shape=ds.shape)\n                    for first in range(0,len(ds),256):a[first:first+256]=ds[first:first+256]\n                    a.flush();del a;os.replace(tmp,folder/(key+'.npy'))\n            json_atomic(marker,identity)\n            print(f'Local cache {fi+1}/{len(records)}: {src.name} ({time.perf_counter()-start:.1f}s)',flush=True)\n        else:\n            if json.loads(marker.read_text())!=identity:raise ValueError('Cache identity mismatch')\n            for key in ['raw_uv','features']:\n                if not (folder/(key+'.npy')).exists():raise ValueError('Incomplete cache: remove its complete.json and retry')\n        self.local_files[path]=folder\n    # Compare actual normalized tensors against the old Drive reader before training.\n    ids=np.unique(np.linspace(0,len(self.samples)-1,min(6,len(self.samples))).astype(int))\n    self.cached_ids=None\n    reference={k:v.copy() for k,v in FullBank._io_original_load(self,ids).items()}\n    self.cached_ids=None\n    actual=_fast_bank_load(self,ids)\n    for k in reference:np.testing.assert_array_equal(reference[k],actual[k])\n    self.cached_ids=None\n    print('Local loader matches the original reader exactly on sampled pairs.',flush=True)\n\ndef _fast_bank_load(self,ids):\n    ids=np.asarray(ids)\n    if self.cached_ids is not None and np.array_equal(ids,self.cached_ids):return self.cached\n    started=time.perf_counter()\n    out={k:np.empty((len(ids),)+self[k].shape[1:],dtype='f4') for k in ['raw','next_raw','spec','next_spec']}\n    groups=defaultdict(list)\n    for j,idx in enumerate(ids):\n        row,i=self.samples[int(idx)];groups[str(self.data.path(row))].append((j,i))\n    s=self.data.scaler\n    for path,entries in groups.items():\n        if path not in self.mmaps:\n            folder=self.local_files[path]\n            self.mmaps[path]=tuple(np.load(folder/(k+'.npy'),mmap_mode='r',allow_pickle=False) for k in ['raw_uv','features'])\n        self.mmaps.move_to_end(path)\n        raw,spec=self.mmaps[path]\n        positions=np.array([j for j,i in entries]);indices=np.array([i for j,i in entries])\n        for key,offset in [('raw',0),('next_raw',1)]:out[key][positions]=(raw[indices+offset]-s['raw_mean'][:,None])/s['raw_std'][:,None]\n        for key,offset in [('spec',0),('next_spec',1)]:out[key][positions]=(spec[indices+offset]-s['feature_mean'])/s['feature_std']\n        while len(self.mmaps)>16:\n            _,old=self.mmaps.popitem(last=False)\n            for a in old:a._mmap.close()\n    if any(not np.isfinite(x).all() for x in out.values()):raise ValueError('Nonfinite training input')\n    self.cached_ids=ids.copy();self.cached=out\n    self.io_seconds+=time.perf_counter()-started;self.io_batches+=1\n    if self.io_batches%250==0:\n        print(f'Loader: {self.io_seconds/self.io_batches:.4f} seconds/batch average (local disk)',flush=True)\n    return out\n\nFullBank.__init__=_fast_bank_init\nFullBank.load=_fast_bank_load\n# Preserve the existing scientific fingerprint for checkpoint compatibility.\n# Record this separately as an explicit I/O implementation change.\njson_atomic(EXPERIMENT/'io_extension.json',dict(version='local_npy_v1',source_sha256=hashlib.sha256(IO_SOURCE.encode()).hexdigest(),\n    parent_implementation=IMPLEMENTATION_SHA,cache=str(LOCAL_TRAINING_CACHE),training_settings_changed=False))\nprint('Local-disk loader enabled. Existing checkpoints in',EXPERIMENT,'will resume.')\n"
exec(compile(IO_SOURCE,"local_training_io.py","exec"))

## 4 · Train, select cutoffs, refit, and score
Each completed outer fold produces outer_results.csv. An interrupted fit resumes from its last checkpoint. Completed prediction files are cached. The outer test is scored only with its saved cutoff. No test metrics are used to change training or thresholds.

In [ ]:
if RUN_EXPERIMENT:
    for outer in chosen:
        for seed in SEEDS:
            run_outer(outer,seed)
else:
    print('Preview complete. Set RUN_EXPERIMENT=True in Section 3, then rerun Sections 3–5.')

## 5 · Collect completed results
Upload completed_outer_results.csv and experiment_plan.json. This table contains one row per outer subject, model and seed; it is not a pooled cohort estimate. Do not average sensitivity over zero-target cases or count repeated seeds as new seizures. A complete study requires every requested outer fold and seed.

In [ ]:
paths=sorted(EXPERIMENT.glob('outer_*/*/outer_results.csv'))
if paths:
    result=pd.concat([pd.read_csv(p) for p in paths],ignore_index=True)
    result.to_csv(EXPERIMENT/'completed_outer_results.csv',index=False)
    display(result)
    print('Saved:',EXPERIMENT/'completed_outer_results.csv')
else: print('No completed outer results yet.')

In [ ]:
# Rescore inner validation predictions only.
# Do not rerun Section 4 for this analysis.

from pathlib import Path
import json
import numpy as np
import pandas as pd

root = EXPERIMENT / 'outer_chb01' / '20261028'
output = root / 'fine_threshold_analysis_v1'
output.mkdir(exist_ok=True)

thresholds = sorted(set(
    [0.95, 1.0, 1.01]
    + np.linspace(0.95, 0.99, 9).tolist()
    + np.linspace(0.99, 0.999, 10).tolist()
    + np.linspace(0.999, 0.9999, 10).tolist()
    + [0.99995, 0.99999, 0.999999]
))

fold = next(f for f in FOLDS if f['id'] == 'outer_chb01')
selections = {}

for variant in VARIANTS:
    reports = {t: [] for t in thresholds}

    for inner in fold['inner_folds']:
        data = FoldData(RUN, 'outer_chb01', inner['id'])
        folder = root / inner['id'] / variant

        predictions = pd.read_csv(folder / 'validation_predictions.csv')
        provenance = json.loads(
            (folder / 'prediction_provenance.json').read_text()
        )

        q, events, coverage = validate_full_prediction_export(
            data, 'validation', predictions, provenance
        )

        for threshold in thresholds:
            _, cases, _, _ = score_cohort(
                q, events, coverage,
                ScoringPolicy(threshold=threshold)
            )
            reports[threshold].append(cases)

        print(variant, inner['id'], 'scored', flush=True)

    best, table = select_threshold(reports)
    table.to_csv(output / f'{variant}_fine_thresholds.csv', index=False)
    selections[variant] = best

    print(variant, best)
    display(table[table.warning <= SETTINGS['warning_cap'] + 1e-12])

(output / 'exploratory_selection.json').write_text(
    json.dumps(selections, indent=2)
)

print('Saved to:', output)

In [ ]:
import json
import pandas as pd

root = EXPERIMENT / 'outer_chb01' / '20261028'
analysis = root / 'fine_threshold_analysis_v1'
output = analysis / 'exploratory_chb01'
output.mkdir(parents=True, exist_ok=True)

# Load thresholds selected from development validation only.
selections = json.loads(
    (analysis / 'exploratory_selection.json').read_text()
)

data = FoldData(RUN, 'outer_chb01', 'outer_refit')
results = []

for variant in VARIANTS:
    folder = root / 'outer_refit' / variant
    predictions = pd.read_csv(folder / 'test_predictions.csv')
    provenance = json.loads(
        (folder / 'prediction_provenance.json').read_text()
    )

    q, events, coverage = validate_full_prediction_export(
        data, 'test', predictions, provenance
    )

    threshold = float(selections[variant]['threshold'])
    pooled, cases, alarms, targets = score_cohort(
        q, events, coverage,
        ScoringPolicy(threshold=threshold)
    )

    destination = output / variant
    destination.mkdir(exist_ok=True)
    cases.to_csv(destination / 'test_cases.csv', index=False)
    alarms.to_csv(destination / 'test_alarms.csv', index=False)
    targets.to_csv(destination / 'test_targets.csv', index=False)

    results.append({
        'outer': 'outer_chb01',
        'seed': 20261028,
        'variant': variant,
        'threshold': threshold,
        'analysis': 'exploratory_fine_thresholds',
        **pooled,
    })
    print(variant, 'scored at threshold', threshold)

results = pd.DataFrame(results)
results.to_csv(output / 'exploratory_outer_results.csv', index=False)

display(results)
print('Saved:', output / 'exploratory_outer_results.csv')

In [ ]:
import json
import numpy as np
import pandas as pd

root = EXPERIMENT / 'outer_chb01' / '20261028'
output = root / 'development_diagnostics_v1'
output.mkdir(exist_ok=True)

fold = next(f for f in FOLDS if f['id'] == 'outer_chb01')
summary, distributions, confusion_rows = [], [], []
prob_columns = ['p_interictal', 'p_preictal', 'p_ictal']

for inner in fold['inner_folds']:
    data = FoldData(RUN, 'outer_chb01', inner['id'])

    for variant in VARIANTS:
        folder = root / inner['id'] / variant
        predictions = pd.read_csv(folder / 'validation_predictions.csv')
        provenance = json.loads(
            (folder / 'prediction_provenance.json').read_text()
        )

        # Check the saved export before analysing it.
        validate_full_prediction_export(
            data, 'validation', predictions, provenance
        )

        metrics = window_metrics(data, predictions)
        summary.append({
            'inner': inner['id'],
            'variant': variant,
            'windows': metrics['ready_labelled_windows'],
            'cross_entropy': metrics['cross_entropy'],
            'balanced_accuracy': metrics['balanced_accuracy'],
            **{
                f'recall_class_{c}': metrics['class_recalls'][c]
                for c in range(3)
            },
        })

        for true_class, counts in enumerate(metrics['confusion_matrix']):
            for predicted_class, count in enumerate(counts):
                confusion_rows.append({
                    'inner': inner['id'], 'variant': variant,
                    'true_class': true_class,
                    'predicted_class': predicted_class,
                    'windows': count,
                })

        labelled = []
        for record in data.records('validation'):
            metadata = data.metadata(record)
            p = predictions[
                predictions.relative_path == record['relative_path']
            ].sort_values('index').copy()

            # Ensure positional label alignment is valid.
            assert np.array_equal(
                p['index'].to_numpy(),
                np.arange(len(metadata['label']))
            )

            p['label'] = metadata['label']
            keep = metadata['prediction_ready'] & (metadata['label'] >= 0)
            labelled.append(p.loc[keep])

        labelled = pd.concat(labelled, ignore_index=True)

        for label, group in labelled.groupby('label'):
            for column in ['p_preictal', 'score']:
                values = group[column].to_numpy(dtype=float)
                if not np.isfinite(values).all():
                    raise ValueError('Nonfinite diagnostic scores')

                distributions.append({
                    'inner': inner['id'], 'variant': variant,
                    'true_class': int(label), 'score_type': column,
                    'windows': len(values),
                    'mean': values.mean(),
                    'q10': np.quantile(values, 0.10),
                    'median': np.median(values),
                    'q90': np.quantile(values, 0.90),
                    'fraction_ge_0_99': np.mean(values >= 0.99),
                })

        print(inner['id'], variant, 'checked', flush=True)

summary = pd.DataFrame(summary)
distributions = pd.DataFrame(distributions)
confusion = pd.DataFrame(confusion_rows)

summary.to_csv(output / 'validation_classification.csv', index=False)
distributions.to_csv(output / 'validation_score_distributions.csv', index=False)
confusion.to_csv(output / 'validation_confusion.csv', index=False)

display(summary)
print('Saved diagnostic files to:', output)

In [ ]:
import numpy as np
import pandas as pd
import torch

root = EXPERIMENT / 'outer_chb01' / '20261028'
data = FoldData(RUN, 'outer_chb01', 'inner_01')

# Load this run's saved classifier.
state = torch.load(
    root / 'inner_01' / 'classifier' / 'last.pt',
    map_location=DEVICE,
    weights_only=False,
)
assert state['variant'] == 'classifier'

model = new_variant(
    'classifier', len(data.cfg['channels']), 20261028, DEVICE
)
model.load_state_dict(state['model'])
model.eval()

# Build the training index without preparing the full local cache.
bank = FullBank.__new__(FullBank)
base_init = getattr(FullBank, '_io_original_init', FullBank.__init__)
base_load = getattr(FullBank, '_io_original_load', FullBank.load)
base_init(bank, data)

rng = np.random.default_rng(20261028)
sample_ids = np.concatenate([
    rng.choice(
        np.flatnonzero(bank['labels'] == c),
        size=min(1000, int((bank['labels'] == c).sum())),
        replace=False,
    )
    for c in range(3)
])
rng.shuffle(sample_ids)

confusion = np.zeros((3, 3), dtype=np.int64)
loss_sum = 0.0
total = 0

with torch.inference_mode():
    for start in range(0, len(sample_ids), 64):
        ids = sample_ids[start:start + 64]
        arrays = base_load(bank, ids)

        raw = torch.as_tensor(
            arrays['raw'], dtype=torch.float32, device=DEVICE
        )
        spec = torch.as_tensor(
            arrays['spec'], dtype=torch.float32, device=DEVICE
        )
        labels = bank['labels'][ids]

        logits, _ = model(raw, spec)
        log_probs = logits.log_softmax(dim=1).cpu().numpy()

        np.add.at(confusion, (labels, log_probs.argmax(axis=1)), 1)
        loss_sum -= log_probs[np.arange(len(labels)), labels].sum()
        total += len(labels)

recalls = confusion.diagonal() / confusion.sum(axis=1)

result = pd.DataFrame([{
    'scope': 'inner_01_training_sample',
    'variant': 'classifier',
    'windows': total,
    'cross_entropy': float(loss_sum / total),
    'balanced_accuracy': float(recalls.mean()),
    **{f'recall_class_{c}': float(recalls[c]) for c in range(3)},
}])

output = root / 'development_diagnostics_v1'
output.mkdir(exist_ok=True)
result.to_csv(output / 'training_sample_classification.csv', index=False)

display(result)
print('Confusion matrix: rows = true class; columns = predicted class')
print(confusion)

In [ ]:
import numpy as np
import pandas as pd
import h5py

data = FoldData(RUN, 'outer_chb01', 'inner_01')
rng = np.random.default_rng(20261028)
rows = []

for role in ['train', 'validation']:
    candidates = {}

    for record in data.records(role):
        m = data.metadata(record)
        ids = np.flatnonzero(
            m['prediction_ready'] & (m['label'] >= 0)
        )
        subject = record['subject']
        candidates.setdefault(subject, []).extend(
            (record, int(i), int(m['label'][i])) for i in ids
        )

    for subject, available in candidates.items():
        if not available:
            continue

        chosen = rng.choice(
            len(available), min(100, len(available)), replace=False
        )
        raw_values, spec_values, labels = [], [], []

        # Group reads by file.
        groups = {}
        for j in chosen:
            record, index, label = available[int(j)]
            groups.setdefault(str(data.path(record)), []).append(
                (index, label)
            )

        for path, entries in groups.items():
            with h5py.File(path, 'r') as h:
                for index, label in entries:
                    raw = (
                        h['raw_uv'][index]
                        - data.scaler['raw_mean'][:, None]
                    ) / data.scaler['raw_std'][:, None]

                    spec = (
                        h['features'][index]
                        - data.scaler['feature_mean']
                    ) / data.scaler['feature_std']

                    raw_values.append(raw.reshape(-1))
                    spec_values.append(spec.reshape(-1))
                    labels.append(label)

        raw = np.concatenate(raw_values)
        spec = np.concatenate(spec_values)

        rows.append({
            'role': role,
            'subject': subject,
            'sampled_windows': len(labels),
            'preictal_sample_fraction': np.mean(np.array(labels) == 1),
            'raw_finite': bool(np.isfinite(raw).all()),
            'spec_finite': bool(np.isfinite(spec).all()),
            'raw_mean_z': raw.mean(),
            'raw_std_z': raw.std(),
            'raw_abs_p99_z': np.quantile(np.abs(raw), .99),
            'raw_fraction_abs_gt_5': np.mean(np.abs(raw) > 5),
            'spec_mean_z': spec.mean(),
            'spec_std_z': spec.std(),
            'spec_abs_p99_z': np.quantile(np.abs(spec), .99),
            'spec_fraction_abs_gt_5': np.mean(np.abs(spec) > 5),
        })
        print(role, subject, 'checked', flush=True)

report = pd.DataFrame(rows)
output = (
    EXPERIMENT / 'outer_chb01' / '20261028'
    / 'development_diagnostics_v1'
)
output.mkdir(exist_ok=True)
report.to_csv(output / 'patient_input_audit.csv', index=False)

display(report)
print('Saved:', output / 'patient_input_audit.csv')

In [ ]:
import json
import numpy as np
import pandas as pd

root = EXPERIMENT / 'outer_chb01' / '20261028'
fold = next(f for f in FOLDS if f['id'] == 'outer_chb01')
results = []

for inner in fold['inner_folds']:
    data = FoldData(RUN, 'outer_chb01', inner['id'])

    for variant in VARIANTS:
        folder = root / inner['id'] / variant
        predictions = pd.read_csv(folder / 'validation_predictions.csv')
        provenance = json.loads(
            (folder / 'prediction_provenance.json').read_text()
        )
        validate_full_prediction_export(
            data, 'validation', predictions, provenance
        )

        patient_confusion = {}

        for record in data.records('validation'):
            m = data.metadata(record)
            p = predictions[
                predictions.relative_path == record['relative_path']
            ].sort_values('index')

            assert np.array_equal(
                p['index'].to_numpy(), np.arange(len(m['label']))
            )

            keep = m['prediction_ready'] & (m['label'] >= 0)
            labels = m['label'][keep].astype(int)
            probs = p[
                ['p_interictal', 'p_preictal', 'p_ictal']
            ].to_numpy()[keep]

            if not np.isfinite(probs).all():
                raise ValueError('Nonfinite probabilities')

            subject = record['subject']
            matrix = patient_confusion.setdefault(
                subject, np.zeros((3, 3), dtype=np.int64)
            )
            np.add.at(matrix, (labels, probs.argmax(axis=1)), 1)

        for subject, matrix in patient_confusion.items():
            counts = matrix.sum(axis=1)
            recalls = np.divide(
                matrix.diagonal(), counts,
                out=np.full(3, np.nan), where=counts > 0
            )

            results.append({
                'inner': inner['id'],
                'variant': variant,
                'subject': subject,
                'windows': int(matrix.sum()),
                'balanced_accuracy_present_classes':
                    float(np.nanmean(recalls)),
                **{
                    f'class_{c}_windows': int(counts[c])
                    for c in range(3)
                },
                **{
                    f'recall_class_{c}': recalls[c]
                    for c in range(3)
                },
                'interictal_predicted_preictal_fraction':
                    matrix[0, 1] / counts[0]
                    if counts[0] else np.nan,
            })

        print(inner['id'], variant, 'checked', flush=True)

report = pd.DataFrame(results)
output = root / 'development_diagnostics_v1'
output.mkdir(exist_ok=True)
path = output / 'per_patient_validation.csv'
report.to_csv(path, index=False)

display(report)
print('Saved:', path)

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

root = EXPERIMENT / 'outer_chb01' / '20261028'
fold = next(f for f in FOLDS if f['id'] == 'outer_chb01')
results = []

for inner in fold['inner_folds']:
    data = FoldData(RUN, 'outer_chb01', inner['id'])

    for variant in VARIANTS:
        folder = root / inner['id'] / variant
        predictions = pd.read_csv(folder / 'validation_predictions.csv')
        provenance = json.loads(
            (folder / 'prediction_provenance.json').read_text()
        )
        validate_full_prediction_export(
            data, 'validation', predictions, provenance
        )

        patients = {}

        for record in data.records('validation'):
            m = data.metadata(record)
            p = predictions[
                predictions.relative_path == record['relative_path']
            ].sort_values('index')

            assert np.array_equal(
                p['index'].to_numpy(), np.arange(len(m['label']))
            )

            # Compare preictal (1) with interictal (0) only.
            keep = m['prediction_ready'] & np.isin(m['label'], [0, 1])
            labels = m['label'][keep].astype(int)
            scores = p['p_preictal'].to_numpy()[keep]

            if not np.isfinite(scores).all():
                raise ValueError('Nonfinite scores')

            patients.setdefault(record['subject'], []).append(
                (labels, scores)
            )

        for subject, parts in patients.items():
            y = np.concatenate([a for a, b in parts])
            score = np.concatenate([b for a, b in parts])
            both_classes = len(np.unique(y)) == 2
            prevalence = float(y.mean()) if len(y) else np.nan
            ap = (
                average_precision_score(y, score)
                if both_classes else np.nan
            )

            results.append({
                'inner': inner['id'],
                'variant': variant,
                'subject': subject,
                'interictal_windows': int((y == 0).sum()),
                'preictal_windows': int((y == 1).sum()),
                'preictal_prevalence': prevalence,
                'auroc': roc_auc_score(y, score)
                    if both_classes else np.nan,
                'average_precision': ap,
                'AP_over_prevalence': ap / prevalence
                    if both_classes else np.nan,
            })

        print(inner['id'], variant, 'checked', flush=True)

report = pd.DataFrame(results)
output = root / 'development_diagnostics_v1'
output.mkdir(exist_ok=True)
path = output / 'per_patient_ranking.csv'
report.to_csv(path, index=False)

display(report.groupby('variant')[
    ['auroc', 'average_precision', 'AP_over_prevalence']
].mean())
print('Saved:', path)

In [ ]:
import numpy as np
import pandas as pd

data = FoldData(RUN, 'outer_chb01', 'inner_01')
counts, mismatches = [], []

for record in data.records('train'):
    m = data.metadata(record)
    starts, ends = m['window_start_s'], m['decision_s']
    cfg = data.cfg

    expected = np.full(len(starts), -1, dtype=np.int8)
    distant = np.ones(len(starts), dtype=bool)
    ictal = np.zeros(len(starts), dtype=bool)
    overlaps = np.zeros(len(starts), dtype=bool)
    post = np.zeros(len(starts), dtype=bool)
    pre = np.zeros(len(starts), dtype=bool)

    for event in data.events:
        if event['case'] != record['case']:
            continue

        onset, offset = event['onset_s'], event['offset_s']
        overlaps |= (starts < offset) & (ends > onset)
        ictal |= (starts >= onset) & (ends <= offset)
        post |= (
            (starts < offset + cfg['postictal_s']) & (ends > offset)
        )
        distant &= (
            (ends <= onset - cfg['interictal_before_s'])
            | (starts >= offset + cfg['postictal_s'])
        )

        if event['eligible_target'] and event['split'] == record['split']:
            pre |= (
                (starts >= onset - cfg['sph_s'] - cfg['sop_s'])
                & (ends <= onset - cfg['sph_s'])
            )

    expected[distant] = 0
    expected[pre & ~post & ~overlaps] = 1
    expected[ictal] = 2

    mismatches.append({
        'subject': record['subject'],
        'file': record['relative_path'],
        'label_mismatches': int((expected != m['label']).sum()),
    })

    next_ok = np.zeros(len(starts), dtype=bool)
    if len(starts) > 1:
        next_ok[:-1] = (
            m['valid_signal'][1:] & m['pair_with_previous'][1:]
        )
    keep = m['prediction_ready'] & (m['label'] >= 0) & next_ok

    for label in range(3):
        counts.append({
            'subject': record['subject'],
            'label': label,
            'training_pairs': int((keep & (m['label'] == label)).sum()),
        })

counts = pd.DataFrame(counts).groupby(
    ['subject', 'label'], as_index=False
).training_pairs.sum()

counts['share_within_class'] = (
    counts.training_pairs
    / counts.groupby('label').training_pairs.transform('sum')
)
# Each class gets one third of balanced training draws.
counts['expected_share_of_training_draws'] = counts.share_within_class / 3

mismatches = pd.DataFrame(mismatches)
output = (
    EXPERIMENT / 'outer_chb01' / '20261028'
    / 'development_diagnostics_v1'
)
output.mkdir(exist_ok=True)
counts.to_csv(output / 'training_patient_balance.csv', index=False)
mismatches.to_csv(output / 'training_label_alignment.csv', index=False)

print('Label mismatches:', mismatches.label_mismatches.sum())
display(counts.sort_values(
    ['label', 'training_pairs'], ascending=[True, False]
))

In [ ]:
import json
import hashlib
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score

data = FoldData(RUN, 'outer_chb01', 'inner_01')
bank = FullBank(data)

destination = RUN / 'sampling_comparison_v1' / 'outer_chb01' / 'inner_01'
destination.mkdir(parents=True, exist_ok=True)

seed = 20261028
steps = 2000
batch_size = 24
largest = max(int((bank['labels'] == c).sum()) for c in range(3))
settings = dict(
    seed=seed,
    steps_per_epoch=int(np.ceil(3 * largest / batch_size)),
    batch_size=batch_size,
)

# Lock this experiment to its settings, inputs, and implementation.
plan = {
    'seed': seed, 'steps': steps, 'batch_size': batch_size,
    'learning_rate': 0.001,
    'implementation': IMPLEMENTATION_SHA,
    'scaler': digest(data.scaler_path),
    'folds': digest(RUN / 'locked_folds.json'),
    'sampling_revision': 1,
}
plan_path = destination / 'plan.json'
if plan_path.exists():
    assert json.loads(plan_path.read_text()) == plan, 'Plan changed'
else:
    plan_path.write_text(json.dumps(plan, indent=2))

fingerprint = hashlib.sha256(
    json.dumps(plan, sort_keys=True).encode()
).hexdigest()

# Available training windows grouped by class and patient.
groups = {}
for index, (record, window) in enumerate(bank.samples):
    label = int(bank['labels'][index])
    groups.setdefault(label, {}).setdefault(
        record['subject'], []
    ).append(index)

groups = {
    c: {s: np.asarray(ids) for s, ids in patients.items()}
    for c, patients in groups.items()
}

def patient_balanced_ids(step):
    rng = np.random.default_rng(seed + 100000 + step)
    ids = []
    for c in range(3):
        subjects = sorted(groups[c])
        for _ in range(batch_size // 3):
            subject = subjects[int(rng.integers(len(subjects)))]
            ids.append(int(rng.choice(groups[c][subject])))
    rng.shuffle(ids)
    return np.asarray(ids)

def save_checkpoint(path, payload):
    temporary = path.with_suffix('.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, path)

models = {}

for arm in ['control', 'patient_balanced']:
    model = new_variant(
        'classifier', len(data.cfg['channels']), seed, DEVICE
    )
    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad],
        lr=0.001,
    )
    checkpoint = destination / f'{arm}_last.pt'
    start = 0
    logs = []

    if checkpoint.exists():
        state = torch.load(
            checkpoint, map_location=DEVICE, weights_only=False
        )
        assert state['fingerprint'] == fingerprint
        assert state['arm'] == arm
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        start, logs = state['step'], state['logs']
        print(arm, 'resuming at', start)

    # Reset the original sampler's cached epoch order.
    bank.epoch = -1
    model.train()

    for step in range(start, steps):
        ids = (
            batch_indices(bank, step, settings)
            if arm == 'control'
            else patient_balanced_ids(step)
        )
        batch = pilot_batch(bank, ids, DEVICE)
        optimizer.zero_grad(set_to_none=True)
        logits, _ = model(batch['raw'], batch['spec'])
        loss = torch.nn.functional.cross_entropy(
            logits, batch['labels']
        )
        if not torch.isfinite(loss):
            raise RuntimeError('Nonfinite training loss')

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        if (step + 1) % 250 == 0 or step + 1 == steps:
            logs.append({
                'step': step + 1,
                'batch_loss': float(loss.detach().cpu()),
            })
            save_checkpoint(checkpoint, {
                'fingerprint': fingerprint, 'arm': arm,
                'step': step + 1, 'model': model.state_dict(),
                'optimizer': optimizer.state_dict(), 'logs': logs,
            })
            print(arm, step + 1, '/', steps, logs[-1], flush=True)

    model.eval()
    models[arm] = model

# Both models evaluated on exactly the same validation windows.
parts = {arm: {} for arm in models}

for number, record in enumerate(data.records('validation'), 1):
    metadata = data.metadata(record)
    probabilities, _ = infer_file(
        list(models.values()), data, record, DEVICE, chunk=128
    )
    keep = (
        metadata['prediction_ready']
        & np.isin(metadata['label'], [0, 1])
    )
    y = metadata['label'][keep].astype(int)

    for j, arm in enumerate(models):
        scores = probabilities[j, keep, 1]
        if not np.isfinite(scores).all():
            raise ValueError('Nonfinite validation scores')
        parts[arm].setdefault(record['subject'], []).append(
            (y, scores)
        )
    if number % 20 == 0:
        print('Validation files checked:', number, flush=True)

rows = []
for arm, patients in parts.items():
    for subject, chunks in patients.items():
        y = np.concatenate([x[0] for x in chunks])
        score = np.concatenate([x[1] for x in chunks])
        both = len(np.unique(y)) == 2
        prevalence = float(y.mean()) if len(y) else np.nan
        ap = average_precision_score(y, score) if both else np.nan
        rows.append({
            'arm': arm, 'subject': subject,
            'preictal_windows': int((y == 1).sum()),
            'interictal_windows': int((y == 0).sum()),
            'auroc': roc_auc_score(y, score) if both else np.nan,
            'average_precision': ap,
            'prevalence': prevalence,
            'AP_over_prevalence': ap / prevalence if both else np.nan,
        })

report = pd.DataFrame(rows)
report.to_csv(destination / 'sampling_comparison.csv', index=False)

display(report)
display(report.groupby('arm')[
    ['auroc', 'average_precision', 'AP_over_prevalence']
].mean())
print('Saved:', destination / 'sampling_comparison.csv')

In [ ]:
import json
import os
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score

data = FoldData(RUN, 'outer_chb01', 'inner_01')
bank = FullBank(data)

source = RUN / 'sampling_comparison_v1' / 'outer_chb01' / 'inner_01'
output = source / 'training_duration_v1'
output.mkdir(exist_ok=True)

plan = json.loads((source / 'plan.json').read_text())
seed = plan['seed']
batch_size = plan['batch_size']
largest = max(int((bank['labels'] == c).sum()) for c in range(3))
settings = {
    'seed': seed,
    'batch_size': batch_size,
    'steps_per_epoch': int(np.ceil(3 * largest / batch_size)),
}

model = new_variant(
    'classifier', len(data.cfg['channels']), seed, DEVICE
)
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=plan['learning_rate'],
)

original = torch.load(
    source / 'control_last.pt',
    map_location=DEVICE, weights_only=False,
)
assert original['step'] == 2000 and original['arm'] == 'control'

resume_path = output / 'last.pt'
state = (
    torch.load(resume_path, map_location=DEVICE, weights_only=False)
    if resume_path.exists() else original
)
assert state['fingerprint'] == original['fingerprint']
assert state['arm'] == 'control'

model.load_state_dict(state['model'])
optimizer.load_state_dict(state['optimizer'])
current = int(state['step'])
bank.epoch = -1
print('Starting/resuming at update', current)

def save_state(path, step):
    temporary = path.with_suffix('.tmp')
    torch.save({
        'fingerprint': original['fingerprint'],
        'arm': 'control', 'step': step,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
    }, temporary)
    os.replace(temporary, path)

@torch.inference_mode()
def evaluate_duration(step):
    model.eval()
    patients = {}

    for record in data.records('validation'):
        m = data.metadata(record)
        probabilities, _ = infer_file(
            [model], data, record, DEVICE, chunk=128
        )
        keep = m['prediction_ready'] & np.isin(m['label'], [0, 1])
        y = m['label'][keep].astype(int)
        scores = probabilities[0, keep, 1]
        if not np.isfinite(scores).all():
            raise ValueError('Nonfinite validation scores')
        patients.setdefault(record['subject'], []).append((y, scores))

    rows = []
    for subject, chunks in patients.items():
        y = np.concatenate([x[0] for x in chunks])
        scores = np.concatenate([x[1] for x in chunks])
        both = len(np.unique(y)) == 2
        prevalence = float(y.mean()) if len(y) else np.nan
        ap = average_precision_score(y, scores) if both else np.nan

        rows.append({
            'step': step, 'subject': subject,
            'auroc': roc_auc_score(y, scores) if both else np.nan,
            'average_precision': ap,
            'prevalence': prevalence,
            'AP_over_prevalence': ap / prevalence if both else np.nan,
        })

    report = pd.DataFrame(rows)
    report.to_csv(output / f'validation_{step}.csv', index=False)
    print('Update', step, 'mean patient AUROC:', report.auroc.mean())
    return report

for target in [5000, 10000]:
    checkpoint = output / f'checkpoint_{target}.pt'
    report_path = output / f'validation_{target}.csv'

    if current < target:
        model.train()
        for step in range(current, target):
            ids = batch_indices(bank, step, settings)
            batch = pilot_batch(bank, ids, DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(batch['raw'], batch['spec'])
            loss = torch.nn.functional.cross_entropy(
                logits, batch['labels']
            )
            if not torch.isfinite(loss):
                raise RuntimeError('Nonfinite loss')
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

            if (step + 1) % 250 == 0:
                save_state(resume_path, step + 1)
                print(step + 1, '/', target,
                      'batch loss', float(loss.detach().cpu()),
                      flush=True)

        current = target

    if current == target:
        save_state(checkpoint, current)
        if not report_path.exists():
            evaluate_duration(target)
    elif not report_path.exists():
        raise RuntimeError(
            f'Missing evaluation at {target}; load its checkpoint separately.'
        )

# Include the existing 2,000-update control result.
baseline = pd.read_csv(source / 'sampling_comparison.csv')
baseline = baseline[baseline.arm == 'control'].copy()
baseline['step'] = 2000

combined = pd.concat([
    baseline,
    pd.read_csv(output / 'validation_5000.csv'),
    pd.read_csv(output / 'validation_10000.csv'),
], ignore_index=True)

combined.to_csv(output / 'duration_comparison.csv', index=False)
display(combined.groupby('step')[
    ['auroc', 'average_precision', 'AP_over_prevalence']
].mean())
print('Saved:', output / 'duration_comparison.csv')

In [ ]:
import json
import hashlib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score, average_precision_score

destination = RUN / 'short_classifier_validation_v1'
destination.mkdir(exist_ok=True)
reports = []

for scope in ['inner_02', 'inner_03']:
    data = FoldData(RUN, 'outer_chb01', scope)
    bank = FullBank(data)
    folder = destination / scope
    folder.mkdir(exist_ok=True)

    largest = max(int((bank['labels'] == c).sum()) for c in range(3))
    settings = dict(
        SETTINGS,
        seed=20261028,
        batch_size=24,
        learning_rate=0.001,
        checkpoint_every=250,
        steps=10000,
        steps_per_epoch=int(np.ceil(3 * largest / 24)),
    )

    plan = {
        'implementation': IMPLEMENTATION_SHA,
        'settings': settings,
        'roles': data.roles,
        'scope': scope,
        'scaler': digest(data.scaler_path),
        'folds': digest(RUN / 'locked_folds.json'),
        'inventory': digest(RUN / 'locked_inventory.json'),
        'events': digest(RUN / 'locked_events.json'),
        'config': digest(RUN / 'config.json'),
    }
    plan_path = folder / 'plan.json'
    if plan_path.exists():
        assert json.loads(plan_path.read_text()) == plan, 'Plan changed'
    else:
        plan_path.write_text(json.dumps(plan, indent=2))

    fingerprint = hashlib.sha256(
        json.dumps(plan, sort_keys=True).encode()
    ).hexdigest()

    # Trains only the classifier; resumes its own checkpoint if present.
    model, _, _ = train_variant(
        'classifier', bank, settings, folder, fingerprint, DEVICE
    )

    patients = {}
    for number, record in enumerate(data.records('validation'), 1):
        m = data.metadata(record)
        probabilities, _ = infer_file(
            [model], data, record, DEVICE, chunk=128
        )
        keep = m['prediction_ready'] & np.isin(m['label'], [0, 1])
        y = m['label'][keep].astype(int)
        scores = probabilities[0, keep, 1]

        if not np.isfinite(scores).all():
            raise ValueError('Nonfinite validation scores')

        patients.setdefault(record['subject'], []).append((y, scores))
        if number % 20 == 0:
            print(scope, 'validation files:', number, flush=True)

    rows = []
    for subject, chunks in patients.items():
        y = np.concatenate([x[0] for x in chunks])
        scores = np.concatenate([x[1] for x in chunks])
        both = len(np.unique(y)) == 2
        prevalence = float(y.mean()) if len(y) else np.nan
        ap = average_precision_score(y, scores) if both else np.nan

        rows.append({
            'scope': scope,
            'subject': subject,
            'steps': 10000,
            'preictal_windows': int((y == 1).sum()),
            'interictal_windows': int((y == 0).sum()),
            'auroc': roc_auc_score(y, scores) if both else np.nan,
            'average_precision': ap,
            'prevalence': prevalence,
            'AP_over_prevalence': ap / prevalence if both else np.nan,
        })

    report = pd.DataFrame(rows)
    report.to_csv(folder / 'validation_ranking.csv', index=False)
    reports.append(report)
    print(scope, 'mean patient AUROC:', report.auroc.mean())

    del model, bank, data
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

combined = pd.concat(reports, ignore_index=True)
path = destination / 'short_classifier_results.csv'
combined.to_csv(path, index=False)

display(combined.groupby('scope')[
    ['auroc', 'average_precision', 'AP_over_prevalence']
].mean())
print('Saved:', path)

In [ ]:
import json
import numpy as np
import pandas as pd
import torch

output = RUN / 'short_classifier_validation_v1' / 'event_scoring_v1'
output.mkdir(parents=True, exist_ok=True)

# Same coarse + fine threshold grid for all three splits.
thresholds = sorted(set(
    np.linspace(0, 0.95, 20).tolist()
    + np.linspace(0.95, 0.99, 9).tolist()
    + np.linspace(0.99, 0.999, 10).tolist()
    + np.linspace(0.999, 0.9999, 10).tolist()
    + [0.99995, 0.99999, 0.999999, 1.0, 1.01]
))
assert SETTINGS['warning_cap'] == 0.20

checkpoints = {
    'inner_01': (
        RUN / 'sampling_comparison_v1' / 'outer_chb01' / 'inner_01'
        / 'training_duration_v1' / 'checkpoint_10000.pt'
    ),
    'inner_02': (
        RUN / 'short_classifier_validation_v1'
        / 'inner_02' / 'classifier' / 'last.pt'
    ),
    'inner_03': (
        RUN / 'short_classifier_validation_v1'
        / 'inner_03' / 'classifier' / 'last.pt'
    ),
}

reports = {threshold: [] for threshold in thresholds}

for scope, checkpoint in checkpoints.items():
    data = FoldData(RUN, 'outer_chb01', scope)
    state = torch.load(
        checkpoint, map_location=DEVICE, weights_only=False
    )
    assert state['step'] == 10000
    if scope == 'inner_01':
        assert state['arm'] == 'control'
    else:
        assert state['variant'] == 'classifier'

    model = new_variant(
        'classifier', len(data.cfg['channels']), 20261028, DEVICE
    )
    model.load_state_dict(state['model'])
    model.eval()

    folder = output / scope
    folder.mkdir(exist_ok=True)
    frames = []

    for number, record in enumerate(data.records('validation'), 1):
        m = data.metadata(record)
        probabilities, scores = infer_file(
            [model], data, record, DEVICE, chunk=128
        )

        frames.append(pd.DataFrame({
            'relative_path': record['relative_path'],
            'index': np.arange(len(m['label'])),
            'case': record['case'],
            'decision_s': m['decision_s'],
            'prediction_ready': m['prediction_ready'],
            'score': scores[0],
            'p_interictal': probabilities[0, :, 0],
            'p_preictal': probabilities[0, :, 1],
            'p_ictal': probabilities[0, :, 2],
        }))

        if number % 20 == 0:
            print(scope, 'validation files:', number, flush=True)

    predictions = pd.concat(frames, ignore_index=True)
    provenance = {
        'purpose': 'full_chronological_predictions',
        'smoke_only': False,
        'outer': data.outer,
        'scope': data.scope,
        'role': 'validation',
        'scaler_sha256': digest(data.scaler_path),
        'folds_sha256': digest(RUN / 'locked_folds.json'),
        'model_checkpoint_sha256': digest(checkpoint),
        'pilot_fingerprint': state['fingerprint'],
        'variant': 'classifier',
    }

    q, events, coverage = validate_full_prediction_export(
        data, 'validation', predictions, provenance
    )
    predictions.to_csv(folder / 'validation_predictions.csv', index=False)
    (folder / 'prediction_provenance.json').write_text(
        json.dumps(provenance, indent=2)
    )

    for threshold in thresholds:
        _, cases, _, _ = score_cohort(
            q, events, coverage,
            ScoringPolicy(threshold=threshold)
        )
        reports[threshold].append(cases)

    print(scope, 'event scoring complete', flush=True)
    del model, data, predictions, frames
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

best, table = select_threshold(reports)
table.to_csv(output / 'short_classifier_thresholds.csv', index=False)
(output / 'development_selection.json').write_text(
    json.dumps(best, indent=2)
)

print('Development-selected operating point:')
print(json.dumps(best, indent=2))
display(table[table.warning <= 0.20 + 1e-12])
print('Saved:', output / 'short_classifier_thresholds.csv')

In [ ]:
import json
import numpy as np
import pandas as pd

output = RUN / 'smoothing_comparison_v1'
output.mkdir(exist_ok=True)

sources = {
    'original': (
        EXPERIMENT / 'outer_chb01' / '20261028'
    ),
    '10000_updates': (
        RUN / 'short_classifier_validation_v1' / 'event_scoring_v1'
    ),
}

# Shared grid includes low thresholds needed by mean smoothing.
thresholds = sorted(set(
    np.linspace(0, 0.95, 96).tolist()
    + np.linspace(0.95, 0.99, 9).tolist()
    + np.linspace(0.99, 0.999, 10).tolist()
    + np.linspace(0.999, 0.9999, 10).tolist()
    + [0.99995, 0.99999, 0.999999, 1.0, 1.01]
))
assert SETTINGS['warning_cap'] == 0.20
summary = []

for duration, source in sources.items():
    reports = {
        method: {t: [] for t in thresholds}
        for method in ['top8', 'mean']
    }

    for scope in ['inner_01', 'inner_02', 'inner_03']:
        data = FoldData(RUN, 'outer_chb01', scope)
        folder = (
            source / scope / 'classifier'
            if duration == 'original' else source / scope
        )
        predictions = pd.read_csv(folder / 'validation_predictions.csv')
        provenance = json.loads(
            (folder / 'prediction_provenance.json').read_text()
        )

        # Validate the original saved export.
        validate_full_prediction_export(
            data, 'validation', predictions, provenance
        )

        by_file = {
            name: frame.sort_values('index')
            for name, frame in predictions.groupby('relative_path')
        }
        mean_frames = []

        for record in data.records('validation'):
            m = data.metadata(record)
            p = by_file[record['relative_path']].copy()
            assert np.array_equal(
                p['index'].to_numpy(), np.arange(len(m['label']))
            )

            history = data.history
            values = p['p_preictal'].to_numpy(dtype=float)
            mean_scores = np.full(len(p), np.nan)

            if len(p) >= history:
                windows = np.lib.stride_tricks.sliding_window_view(
                    values, history
                )
                mean_scores[history - 1:] = windows.mean(axis=1)

                # Verify our reconstruction matches existing Top-8 fusion.
                reconstructed = np.full(len(p), np.nan)
                reconstructed[history - 1:] = np.partition(
                    windows, history - 8, axis=1
                )[:, -8:].mean(axis=1)

                ready = m['prediction_ready']
                assert np.allclose(
                    reconstructed[ready],
                    p['score'].to_numpy()[ready],
                    atol=2e-7, rtol=1e-6,
                ), 'Top-8 reconstruction differs'

            mean_scores[~m['prediction_ready']] = np.nan
            if not np.isfinite(
                mean_scores[m['prediction_ready']]
            ).all():
                raise ValueError('Missing ready mean scores')

            p['score'] = mean_scores
            mean_frames.append(p)

        mean_predictions = pd.concat(mean_frames, ignore_index=True)

        for method, frame in [
            ('top8', predictions), ('mean', mean_predictions)
        ]:
            q, events, coverage = validate_full_prediction_export(
                data, 'validation', frame, provenance
            )

            for threshold in thresholds:
                _, cases, _, _ = score_cohort(
                    q, events, coverage,
                    ScoringPolicy(threshold=threshold)
                )
                reports[method][threshold].append(cases)

            print(duration, scope, method, 'scored', flush=True)

    for method in ['top8', 'mean']:
        best, table = select_threshold(reports[method])
        table.to_csv(
            output / f'{duration}_{method}_thresholds.csv',
            index=False,
        )

        selected_cases = pd.concat(
            reports[method][best['threshold']], ignore_index=True
        )
        summary.append({
            'training': duration,
            'smoothing': method,
            **best,
            'targets': int(selected_cases.targets.sum()),
            'predicted_targets':
                int(selected_cases.predicted_targets.sum()),
        })

result = pd.DataFrame(summary)
result.to_csv(output / 'smoothing_comparison.csv', index=False)
display(result)
print('Saved:', output / 'smoothing_comparison.csv')

In [ ]:
import json
import hashlib
import numpy as np
import pandas as pd
import torch

SEED_CHECK = 20261029
destination = (
    RUN / 'short_classifier_seed_check_v1' / str(SEED_CHECK)
)
destination.mkdir(parents=True, exist_ok=True)

thresholds = sorted(set(
    np.linspace(0, 0.95, 96).tolist()
    + np.linspace(0.95, 0.99, 9).tolist()
    + np.linspace(0.99, 0.999, 10).tolist()
    + np.linspace(0.999, 0.9999, 10).tolist()
    + [0.99995, 0.99999, 0.999999, 1.0, 1.01]
))
assert SETTINGS['warning_cap'] == 0.20

reports = {
    method: {t: [] for t in thresholds}
    for method in ['top8', 'mean']
}

for scope in ['inner_01', 'inner_02', 'inner_03']:
    data = FoldData(RUN, 'outer_chb01', scope)
    bank = FullBank(data)
    folder = destination / scope
    folder.mkdir(exist_ok=True)

    largest = max(int((bank['labels'] == c).sum()) for c in range(3))
    settings = dict(
        SETTINGS,
        seed=SEED_CHECK,
        steps=10000,
        batch_size=24,
        learning_rate=0.001,
        checkpoint_every=250,
        steps_per_epoch=int(np.ceil(3 * largest / 24)),
    )

    plan = {
        'implementation': IMPLEMENTATION_SHA,
        'settings': settings,
        'scope': scope,
        'roles': data.roles,
        'thresholds': thresholds,
        'smoothing': ['top8', 'mean'],
        'scaler': digest(data.scaler_path),
        'folds': digest(RUN / 'locked_folds.json'),
        'inventory': digest(RUN / 'locked_inventory.json'),
        'events': digest(RUN / 'locked_events.json'),
        'config': digest(RUN / 'config.json'),
    }
    plan_path = folder / 'plan.json'
    if plan_path.exists():
        assert json.loads(plan_path.read_text()) == plan, 'Plan changed'
    else:
        plan_path.write_text(json.dumps(plan, indent=2))

    fingerprint = hashlib.sha256(
        json.dumps(plan, sort_keys=True).encode()
    ).hexdigest()

    model, _, _ = train_variant(
        'classifier', bank, settings, folder, fingerprint, DEVICE
    )

    checkpoint = folder / 'classifier' / 'last.pt'
    prediction_path = folder / 'validation_predictions.csv'
    provenance_path = folder / 'prediction_provenance.json'

    provenance = {
        'purpose': 'full_chronological_predictions',
        'smoke_only': False,
        'outer': data.outer,
        'scope': data.scope,
        'role': 'validation',
        'scaler_sha256': digest(data.scaler_path),
        'folds_sha256': digest(RUN / 'locked_folds.json'),
        'model_checkpoint_sha256': digest(checkpoint),
        'pilot_fingerprint': fingerprint,
        'variant': 'classifier',
    }

    if prediction_path.exists() and provenance_path.exists():
        assert json.loads(provenance_path.read_text()) == provenance
        predictions = pd.read_csv(prediction_path)
    else:
        frames = []
        for number, record in enumerate(data.records('validation'), 1):
            m = data.metadata(record)
            probabilities, scores = infer_file(
                [model], data, record, DEVICE, chunk=128
            )
            frames.append(pd.DataFrame({
                'relative_path': record['relative_path'],
                'index': np.arange(len(m['label'])),
                'case': record['case'],
                'decision_s': m['decision_s'],
                'prediction_ready': m['prediction_ready'],
                'score': scores[0],
                'p_interictal': probabilities[0, :, 0],
                'p_preictal': probabilities[0, :, 1],
                'p_ictal': probabilities[0, :, 2],
            }))
            if number % 20 == 0:
                print(scope, 'validation files:', number, flush=True)

        predictions = pd.concat(frames, ignore_index=True)
        validate_full_prediction_export(
            data, 'validation', predictions, provenance
        )
        predictions.to_csv(prediction_path, index=False)
        provenance_path.write_text(json.dumps(provenance, indent=2))

    validate_full_prediction_export(
        data, 'validation', predictions, provenance
    )

    mean_frames = []
    for _, frame in predictions.groupby('relative_path'):
        frame = frame.sort_values('index').copy()
        values = frame['p_preictal'].to_numpy(dtype=float)
        ready = frame['prediction_ready'].to_numpy(dtype=bool)
        scores = np.full(len(frame), np.nan)
        history = data.history

        if len(frame) >= history:
            windows = np.lib.stride_tricks.sliding_window_view(
                values, history
            )
            scores[history - 1:] = windows.mean(axis=1)

        scores[~ready] = np.nan
        if not np.isfinite(scores[ready]).all():
            raise ValueError('Missing ready mean scores')
        frame['score'] = scores
        mean_frames.append(frame)

    mean_predictions = pd.concat(mean_frames, ignore_index=True)

    for method, frame in [
        ('top8', predictions), ('mean', mean_predictions)
    ]:
        q, events, coverage = validate_full_prediction_export(
            data, 'validation', frame, provenance
        )
        for threshold in thresholds:
            _, cases, _, _ = score_cohort(
                q, events, coverage,
                ScoringPolicy(threshold=threshold)
            )
            reports[method][threshold].append(cases)

        print(scope, method, 'scored', flush=True)

    del model, bank, data, predictions, mean_predictions
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary = []
for method in ['top8', 'mean']:
    best, table = select_threshold(reports[method])
    table.to_csv(
        destination / f'{method}_thresholds.csv', index=False
    )
    cases = pd.concat(
        reports[method][best['threshold']], ignore_index=True
    )
    summary.append({
        'seed': SEED_CHECK,
        'smoothing': method,
        **best,
        'targets': int(cases.targets.sum()),
        'predicted_targets': int(cases.predicted_targets.sum()),
    })

result = pd.DataFrame(summary)
result.to_csv(destination / 'seed_comparison.csv', index=False)
display(result)
print('Saved to:', destination)

In [ ]:
import json
import hashlib
import numpy as np
import pandas as pd
import torch

destination = RUN / 'short_three_model_mean_v1'
destination.mkdir(exist_ok=True)

thresholds = sorted(set(
    np.linspace(0, 0.95, 96).tolist()
    + np.linspace(0.95, 0.99, 9).tolist()
    + np.linspace(0.99, 0.999, 10).tolist()
    + np.linspace(0.999, 0.9999, 10).tolist()
    + [0.99995, 0.99999, 0.999999, 1.0, 1.01]
))
assert SETTINGS['warning_cap'] == 0.20

def classifier_checkpoint(seed, scope):
    if seed == 20261029:
        return (
            RUN / 'short_classifier_seed_check_v1' / str(seed)
            / scope / 'classifier' / 'last.pt'
        )
    if scope == 'inner_01':
        return (
            RUN / 'sampling_comparison_v1' / 'outer_chb01' / scope
            / 'training_duration_v1' / 'checkpoint_10000.pt'
        )
    return (
        RUN / 'short_classifier_validation_v1'
        / scope / 'classifier' / 'last.pt'
    )

all_summary = []

for seed in [20261028, 20261029]:
    reports = {
        v: {t: [] for t in thresholds}
        for v in ['classifier', 'koopman', 'physics']
    }

    for scope in ['inner_01', 'inner_02', 'inner_03']:
        data = FoldData(RUN, 'outer_chb01', scope)
        bank = FullBank(data)
        folder = destination / str(seed) / scope
        folder.mkdir(parents=True, exist_ok=True)

        largest = max(
            int((bank['labels'] == c).sum()) for c in range(3)
        )
        settings = dict(
            SETTINGS, seed=seed, steps=10000,
            batch_size=24, learning_rate=0.001,
            checkpoint_every=250,
            steps_per_epoch=int(np.ceil(3 * largest / 24)),
        )

        checkpoint = classifier_checkpoint(seed, scope)
        state = torch.load(
            checkpoint, map_location=DEVICE, weights_only=False
        )
        assert state['step'] == 10000
        if seed == 20261028 and scope == 'inner_01':
            assert state['arm'] == 'control'
        else:
            assert state['variant'] == 'classifier'

        plan = {
            'implementation': IMPLEMENTATION_SHA,
            'settings': settings,
            'roles': data.roles,
            'scope': scope,
            'classifier_checkpoint': digest(checkpoint),
            'scaler': digest(data.scaler_path),
            'folds': digest(RUN / 'locked_folds.json'),
            'inventory': digest(RUN / 'locked_inventory.json'),
            'events': digest(RUN / 'locked_events.json'),
            'config': digest(RUN / 'config.json'),
            'smoothing': 'causal_mean_existing_history',
            'thresholds': thresholds,
        }
        plan_path = folder / 'plan.json'
        if plan_path.exists():
            assert json.loads(plan_path.read_text()) == plan, 'Plan changed'
        else:
            plan_path.write_text(json.dumps(plan, indent=2))

        fingerprint = hashlib.sha256(
            json.dumps(plan, sort_keys=True).encode()
        ).hexdigest()

        classifier = new_variant(
            'classifier', len(data.cfg['channels']), seed, DEVICE
        )
        classifier.load_state_dict(state['model'])
        models = {'classifier': classifier}
        checkpoint_paths = {'classifier': checkpoint}

        for variant in ['koopman', 'physics']:
            bank.epoch = -1
            models[variant] = train_variant(
                variant, bank, settings, folder, fingerprint, DEVICE
            )[0]
            checkpoint_paths[variant] = folder / variant / 'last.pt'

        names = list(models)
        prediction_frames = {v: [] for v in names}

        for number, record in enumerate(data.records('validation'), 1):
            m = data.metadata(record)
            probabilities, _ = infer_file(
                list(models.values()), data, record, DEVICE, chunk=128
            )

            for j, variant in enumerate(names):
                values = probabilities[j, :, 1]
                scores = np.full(len(values), np.nan)
                history = data.history

                if len(values) >= history:
                    windows = np.lib.stride_tricks.sliding_window_view(
                        values, history
                    )
                    scores[history - 1:] = windows.mean(axis=1)

                scores[~m['prediction_ready']] = np.nan
                if not np.isfinite(scores[m['prediction_ready']]).all():
                    raise ValueError('Missing ready mean scores')

                prediction_frames[variant].append(pd.DataFrame({
                    'relative_path': record['relative_path'],
                    'index': np.arange(len(values)),
                    'case': record['case'],
                    'decision_s': m['decision_s'],
                    'prediction_ready': m['prediction_ready'],
                    'score': scores,
                    'p_interictal': probabilities[j, :, 0],
                    'p_preictal': probabilities[j, :, 1],
                    'p_ictal': probabilities[j, :, 2],
                }))

            if number % 20 == 0:
                print(seed, scope, 'validation files:', number, flush=True)

        for variant in names:
            p = pd.concat(prediction_frames[variant], ignore_index=True)
            provenance = {
                'purpose': 'full_chronological_predictions',
                'smoke_only': False,
                'outer': data.outer, 'scope': scope,
                'role': 'validation',
                'variant': variant,
                'scaler_sha256': digest(data.scaler_path),
                'folds_sha256': digest(RUN / 'locked_folds.json'),
                'model_checkpoint_sha256':
                    digest(checkpoint_paths[variant]),
                'pilot_fingerprint': (
                    state['fingerprint']
                    if variant == 'classifier' else fingerprint
                ),
                'smoothing': 'causal_mean_existing_history',
            }
            q, events, coverage = validate_full_prediction_export(
                data, 'validation', p, provenance
            )

            export_folder = folder / variant
            export_folder.mkdir(exist_ok=True)
            p.to_csv(
                export_folder / 'mean_validation_predictions.csv',
                index=False,
            )
            (export_folder / 'mean_prediction_provenance.json').write_text(
                json.dumps(provenance, indent=2)
            )

            for threshold in thresholds:
                _, cases, _, _ = score_cohort(
                    q, events, coverage,
                    ScoringPolicy(threshold=threshold)
                )
                reports[variant][threshold].append(cases)

            print(seed, scope, variant, 'scored', flush=True)

        del models, classifier, bank, data, prediction_frames, state
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    for variant, threshold_reports in reports.items():
        best, table = select_threshold(threshold_reports)
        table.to_csv(
            destination / str(seed) / f'{variant}_thresholds.csv',
            index=False,
        )
        cases = pd.concat(
            threshold_reports[best['threshold']], ignore_index=True
        )
        cases.to_csv(
            destination / str(seed) / f'{variant}_selected_cases.csv',
            index=False,
        )
        all_summary.append({
            'seed': seed, 'variant': variant,
            **best,
            'targets': int(cases.targets.sum()),
            'predicted_targets': int(cases.predicted_targets.sum()),
        })

    # Save after each completed seed.
    pd.DataFrame(all_summary).to_csv(
        destination / 'three_model_comparison.csv', index=False
    )

display(pd.DataFrame(all_summary))
print('Saved:', destination / 'three_model_comparison.csv')

In [ ]:
import numpy as np
import pandas as pd

root = RUN / 'short_three_model_mean_v1'
frames = []

for seed in [20261028, 20261029]:
    for variant in ['classifier', 'koopman', 'physics']:
        frame = pd.read_csv(
            root / str(seed) / f'{variant}_selected_cases.csv'
        )
        frame['seed'] = seed
        frame['variant'] = variant
        frames.append(frame)

cases = pd.concat(frames, ignore_index=True)

# Sensitivity is defined only for patients with eligible targets.
eligible = cases[cases.targets > 0].copy()
assert not eligible.duplicated(['case', 'seed', 'variant']).any()

# Average the two seed results within each patient.
# Seeds are not treated as additional independent patients.
patient = eligible.pivot(
    index='case', columns=['variant', 'seed'], values='sensitivity'
)
assert not patient.isna().any().any(), 'Unmatched patients or seeds'

scores = pd.DataFrame({
    variant: patient[variant].mean(axis=1)
    for variant in ['classifier', 'koopman', 'physics']
})

rng = np.random.default_rng(20261101)
draws = rng.integers(
    0, len(scores), size=(10000, len(scores))
)

rows = []
for first, second in [
    ('koopman', 'classifier'),
    ('physics', 'classifier'),
    ('physics', 'koopman'),
]:
    differences = (scores[first] - scores[second]).to_numpy()
    bootstrap = differences[draws].mean(axis=1)
    low, high = np.quantile(bootstrap, [0.025, 0.975])

    rows.append({
        'comparison': f'{first} minus {second}',
        'patients': len(scores),
        'mean_difference_percentage_points':
            100 * differences.mean(),
        'bootstrap_95_low_percentage_points': 100 * low,
        'bootstrap_95_high_percentage_points': 100 * high,
    })

report = pd.DataFrame(rows)
output = root / 'descriptive_uncertainty_v1'
output.mkdir(exist_ok=True)
scores.to_csv(output / 'patient_seed_mean_sensitivity.csv')
report.to_csv(output / 'paired_model_uncertainty.csv', index=False)

display(report)
print('Saved:', output / 'paired_model_uncertainty.csv')

In [ ]:
import io
import json
import zipfile
import numpy as np
import pandas as pd

root = RUN / 'short_three_model_mean_v1'
archive = root / 'chance_analysis_inputs.zip'

with zipfile.ZipFile(
    archive, 'w', compression=zipfile.ZIP_DEFLATED
) as bundle:
    for seed in [20261028, 20261029]:
        for scope in ['inner_01', 'inner_02', 'inner_03']:
            data = FoldData(RUN, 'outer_chb01', scope)

            for variant in ['classifier', 'koopman', 'physics']:
                folder = root / str(seed) / scope / variant
                predictions = pd.read_csv(
                    folder / 'mean_validation_predictions.csv'
                )
                provenance = json.loads(
                    (folder / 'mean_prediction_provenance.json').read_text()
                )
                q, events, coverage = validate_full_prediction_export(
                    data, 'validation', predictions, provenance
                )

                prefix = f'{seed}/{scope}/{variant}'
                buffer = io.BytesIO()
                np.savez_compressed(
                    buffer,
                    case=q['case'].to_numpy(dtype=str),
                    relative_path=q['relative_path'].to_numpy(dtype=str),
                    index=q['index'].to_numpy(dtype=np.int64),
                    decision_s=q['decision_s'].to_numpy(dtype=float),
                    prediction_ready=q['prediction_ready'].to_numpy(
                        dtype=bool
                    ),
                    score=q['score'].to_numpy(dtype=float),
                )
                bundle.writestr(prefix + '/predictions.npz',
                                buffer.getvalue())
                bundle.writestr(prefix + '/provenance.json',
                                json.dumps(provenance))

                if seed == 20261028 and variant == 'classifier':
                    bundle.writestr(
                        f'{scope}/events.json',
                        events.to_json(orient='records', double_precision=15),
                    )
                    bundle.writestr(
                        f'{scope}/coverage.json',
                        json.dumps({
                            case: [[float(a), float(b)] for a, b in spans]
                            for case, spans in coverage.items()
                        }),
                    )

            print(seed, scope, 'exported', flush=True)

        for variant in ['classifier', 'koopman', 'physics']:
            path = root / str(seed) / f'{variant}_thresholds.csv'
            bundle.write(path, f'{seed}/{path.name}')

    bundle.write(
        root / 'three_model_comparison.csv',
        'three_model_comparison.csv',
    )

print('Upload:', archive)
print('Size:', round(archive.stat().st_size / 2**20, 1), 'MB')